In [ ]:
Grad_boost = GBoost(classifier = 'MLP', n_estimators = 5, learning_rate = 0.5, seed = 21, max_depth = 3, optimiser = 'SGD',
                    objective = 'MSE', loss_function = 'MSE', decay=0.01, decay_function = 'beta', decay_steps = 2, epsilon3= 0.001, reg_alpha = 0.3,
                    distribution = st.beta.pdf(np.linspace(0, 1, 50), 2, 3, loc= 0, scale = 1.5))
Grad_boost.fit(X_train, y_train)

plot_GB_res(X_train, y_train, Grad_boost._predict_train(X_train), 'Training')
plot_GB_res(X_test, y_test, Grad_boost.predict(X_test), "Testing")

print(f"Training Mean squared Error: {metrics.mean_squared_error(y_train, Grad_boost._predict_train(X_train)):.4f}")
print(f"Testing Mean squared Error: {metrics.mean_squared_error(y_test, Grad_boost.predict(X_test)):.4f}")

In [ ]:
def _fit_MLP(self, X, y):
        mlp = MLP(self.optimiser, self.hidden_layer_sizes, self.activation, self.loss_function, 
                  self.beta2, self.beta3, self.alpha, self.epsilon2, self.n_epochs, self.batch_size,
                  self.beta, self.gamma, self.seed) #self.n_epochs
        mlp.fit(X, y)
        self.models.append(mlp)  
        y_preds = self.learning_rate * mlp.predict(X)
        self.estimator_preds.append(y_preds)
        self.current_pred = y_preds
#         self.current_pred += self.learning_rate * mlp.predict(X)
#         self.estimator_preds.append(self.current_pred)
        return self    

 def _fit_xg(self, X, y):
        gradients, hessians = self._objective(y, self.current_pred)
        sample_idxs = None if self.subsample == 1.0 else self._get_subsample(y)
        booster = TreeBooster(X, gradients, hessians, self.min_child_weight, self.reg_gamma, self.reg_lambda, self.reg_alpha, self.colsample_bynode, self.max_depth, sample_idxs)    
        self.models.append(booster)
        y_preds = self.learning_rate * booster.predict(X)
        self.estimator_preds.append(y_preds)
        self.current_pred = y_preds
        return self   

In [ ]:
%autoreload
class GBoost(BaseEstimator, RegressorMixin):
    def __init__(self, classifier = 'stump', n_estimators = 10, learning_rate = 0.5, max_depth = 1, 
                 subsample = 1, gamma = 0.5, beta = 0.5, min_samples_leaf = 0, min_samples_split = 0, 
                 loss_function = 'MSE', validation_fraction = None, patience = None, tolerance = None,
                 hidden_layer_sizes=(10,), activation='relu', optimiser = None,
                 beta2 =0.9, beta3 = 0.999, epsilon2 = 1e-8, eta2=0.01, n_epochs=1000, batch_size=None,
                 seed = None, measure = 'MSE', drop_rate = 0,
                 base_prediction = 0.0, delta = 1.0, tau = 1, objective = None, colsample_bynode = 1, reg_gamma = 0, reg_lambda = 1.0,
                 min_child_weight = 1.0, reg_alpha = 1.0,
                 decay_function = None, distribution = None, decay = None, decay_steps = 1, power = 1, epsilon3 = 1e-8):
        
        self.classifier = classifier
        self.models = []
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.gamma = gamma #for huber loss
        self.beta = beta #for huber loss
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.subsample = subsample
        self.loss_function = loss_function
        self.seed = seed
        self.measure = measure
#         self.stumps = []
        self.loss_history = []  #records loss for plotting
        self.initial_pred = None
        self.current_pred = None
        self.root = None
#         self.trees = []
        #subsample params
        self.validation_fraction = validation_fraction
        self.validation_scores = []
        self.best_val_score = float('inf')
        self.patience = patience #(no iteration change)
        self.tolerance = tolerance
        self.best_iteration = 0
        self.no_improvement_count = 0
        #learning schedule params
        self.distribution = distribution                  #a statistical distribution for decay schedule
#         self.distribution = None
        self.eta_history = []
        self.eta_testing_history = []
        self.decay = decay
        self.decay_steps = decay_steps
        self.power = power
        self.decay_function = decay_function
        self.estimator_preds = [] #training preds based on current estimator
        self.testing_preds = [] #round by round predictions on test data for post hoc analysis
        self.initial_learning_rate = self.learning_rate #for exponential decay
        self.eta = [] #store list of size y, with the learning rate. For adjusting learning rate dynamically on each sample
        self.G = [] #for adaptive decay
        self.epsilon3 = epsilon3 #modified adative gradient
        self.test_eta = []
        #MLP params
        self.hidden_layer_sizes = hidden_layer_sizes
        self.optimiser = optimiser
        self.activation = activation
        self.eta2 = eta2
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.weights = []
        self.biases = []
        self.delta_history = []
        self.weights_history = []
        self.biases_history = []
        #MLP adam
        self.beta2 = beta2
        self.beta3 = beta3
        self.epsilon2 = epsilon2
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = [] 
        #XGB params
        self.base_prediction = base_prediction
        self.delta = delta
        self.tau = tau #in huber objective
        self.reg_gamma = reg_gamma #controls tree pruning regularisation
        self.reg_lambda = reg_lambda #L2 regularisation
        self.reg_alpha = reg_alpha #L1 penalisation score
        self.min_child_weight = min_child_weight #controls tree splitting
        self.colsample_bynode = colsample_bynode
        self.objective = objective
        self.boosters = []
        #D.A.R.T params
        self.drop_rate = drop_rate
        self.DART_pred = None
        self.active_models = []
        self.DART_models = []
        
        # Set random seed if provided
        if self.seed is not None:
            np.random.seed(self.seed)
        
    def _compute_residuals(self, y, y_pred): 
#         print(f'_compute_residuals: len y: {len(y)}, len y_pred {len(y_pred)}')
        if self.loss_function is 'huber': return self._huber_gradient((y - y_pred))
        else: return y - y_pred
        
    def _get_subsample(self, y): #return indices of sub sampled data
        np.random.seed(self.seed)
        return np.array(np.random.choice(len(y), size=(round(len(y) * self.subsample)), replace=False))
    
    def _initialize_predictions(self, y): #also use for leaf value
        if self.loss_function is 'MAE': return np.median(y)
        elif self.loss_function is 'huber': return self._huber_initialise_predictions(y)
        else: return y.mean()   
    
    def _huber_initialise_predictions(self, y): #to start the preds
        best_loss = float('inf')
        ip = None
        for i in y:
            y_t = self._huber_loss(i) #loss at this point
            total_loss = self._huber_loss(y - y_t) #loss across all data using that point
            if total_loss < best_loss: #minimise loss over data
                best_loss = total_loss
                ip = i                 #get minimising initial prediction         
        return ip
    
    #=======================================LOSS FUNCTIONS=======================================
    def _huber_gradient(self, residuals): #transforms residuals inot a huber gradient. #change epsilon to gamma
        abs_resid = np.abs(residuals)
        gradient = np.where(abs_resid <= self.gamma, -residuals, -self.gamma * np.sign(residuals))
        return gradient
    
    def _huber_loss(self, residuals): #to find leaf values with huber loss minimisation
        abs_resid = np.abs(residuals)
        quadratic = self.beta * residuals**2
        linear = self.gamma * (abs_resid - 0.5 * self.gamma)
        return np.where(abs_resid <= self.gamma, quadratic, linear).sum()
    
    def _mse_(self, residuals, left_mask, right_mask):
        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        mse = (np.mean((residuals[left_mask] - left_value) ** 2) * np.sum(left_mask) + 
           np.mean((residuals[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
        return mse, left_value, right_value

    def _abs_error(self, residuals, left_mask, right_mask): #note, residuals have to be median, not mean for abs_error: outliers
        sign_vector = np.sign(residuals)

        left_value = sign_vector[left_mask].mean()
        right_value = sign_vector[right_mask].mean()

        mae = (np.mean((sign_vector[left_mask] - left_value) ** 2) * np.sum(left_mask) +
               np.mean((sign_vector[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)

        return mae, left_value, right_value

    def _friedman_MSE(self, residuals, left_mask, right_mask, weights = None):
        if weights is None:
            weights = np.ones(len(residuals))

        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        w_l = np.sum(weights[left_mask]) 
        w_r = np.sum(weights[right_mask])

        mean_l = np.mean(residuals[left_mask])
        mean_r = np.mean(residuals[right_mask])

        friedman_mse = ((w_l * w_r) / (w_l + w_r)) * ((mean_l - mean_r)**2)

        return friedman_mse, left_value, right_value

    def _huber(self, residuals, left_mask, right_mask):
        huber_residuals = self._huber_gradient(residuals) #always huber-ise the residuals, even when already hubered from r-1 

        left_value = np.mean(huber_residuals[left_mask]) if np.any(left_mask) else 0
        right_value = np.mean(huber_residuals[right_mask]) if np.any(right_mask) else 0
        #need to figure out best huber loss over all predictions. 
        y_pred = np.full_like(huber_residuals, left_value)
        y_pred[right_mask] = right_value
        
        error = self._huber_loss(self._compute_residuals(huber_residuals, y_pred))
        
        return error, left_value, right_value   
    
    #=======================================STUMP FUNCTIONS=======================================
    def _train_weak_learner(self, X, residuals):
        X = np.array(X)  # Ensure X is a NumPy array
        n_samples, n_features = X.shape
        best_feature_index = None
        best_threshold = None
        best_left_value = None
        best_right_value = None

        if self.loss_function is 'friedmanMSE': best_error = float('-inf')
        else: best_error = float('inf')

        # Iterate over each feature
        for feature_index in range(n_features):
            thresholds = np.unique(X[:, feature_index])

            # Test each threshold as a potential split point
            for threshold in thresholds:
                left_mask = X[:, feature_index] <= threshold
                right_mask = X[:, feature_index] > threshold

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                    
                #skip if either side has fewer samples than the specified minimum samples per leaf
                if (np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf): continue
                    
                #impurity calculation
                if self.loss_function is 'MSE': error, left_value, right_value =  self._mse_(residuals, left_mask, right_mask)
                elif self.loss_function is 'friedmanMSE': error, left_value, right_value =  self._friedman_MSE(residuals, left_mask, right_mask)
                elif self.loss_function is 'MAE': error, left_value, right_value =  self._abs_error(residuals, left_mask, right_mask)
                elif self.loss_function is 'huber': error, left_value, right_value = self._huber(residuals, left_mask, right_mask)

                    
    #             If this is the best split so far, store it. FreidmanMSE maximises, so we need a different approach
                if self.loss_function is 'friedmanMSE':
                    if error > best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value
                else:
                    if error < best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value

        return best_feature_index, best_threshold, best_left_value, best_right_value, best_error
    
    def _fit_stump(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            stump = self._train_weak_learner(X[indices], resid_subsample)
            feature_index, threshold, left_value, right_value, best_error = stump

            current_pred_subsample = self.current_pred[indices]
            updates = self._update_predictions(current_pred_subsample, feature_index, threshold, left_value, right_value, X[indices])
            self.current_pred[indices] = updates
            #========================
        else: 
            stump = self._train_weak_learner(X, residuals)
            feature_index, threshold, left_value, right_value, best_error = stump
            self.current_pred = self._update_predictions(self.current_pred, feature_index, threshold, left_value, right_value, X)
        #========================
        self.models.append(stump)
        self.estimator_preds.append(self.current_pred)
        self.loss_history.append(best_error)
        return self
    
    #=======================================TREE FUNCTIONS=======================================
    def _leaf_value(self, y): return np.mean(y) #y is residuals for GBoost, can use initialise because the prediction is the mean
    
    def _grow_tree(self, X, residuals, depth = 0): #recursively grow the decision tree up to the specified depth. Which is a stump if not specified
#       #we will recursively get subsamples of the branches to stop braodcasting error. If subsample =1, we use all data
        indices = self._get_subsample(residuals) #get indices
        residuals = residuals[indices]
        X = X[indices]
        n_samples, n_features = X.shape
        
        #check stopping criteria
        if (depth >= self.max_depth or n_samples < self.min_samples_split): 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        #find best split
        best_feature, best_threshold, left_value, right_value, best_error = self._train_weak_learner(X, residuals)
        
        #if no split found, create a leaf
        if best_feature is None: 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        y_pred =  self.current_pred[indices]
        y_pred = self._update_predictions(y_pred, best_feature, best_threshold, left_value, right_value, X)
        self.current_pred[indices] = y_pred
#       self.current_pred = self._update_predictions(self.current_pred, best_feature, best_threshold, left_value, right_value, X)
        
        #split data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold
        left_subtree = self._grow_tree(X[left_mask], residuals[left_mask], depth +1)
        right_subtree = self._grow_tree(X[right_mask], residuals[right_mask], depth +1)
                
        #track the tree stats
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'left value': left_value,
            'right value': right_value,
            'left': left_subtree,
            'right': right_subtree,
            'error': best_error,
            'is_leaf': False,
            'predictions':self.current_pred
        }
    
    def _fit_tree(self, t, X, residuals): #updates with learning rate are called in _grow_tree . _update_predictions
        self.root = self._grow_tree(X, residuals)
#         tree = {}
#         tree = []
#         tree[f'tree {t}'] = self.root
#         tree[] = self.root
        self.loss_history.append(self.root['error'])
#         self.models.append(tree)
        self.models.append(self.root)
        self.estimator_preds.append(self.current_pred)
        return self
            
    #=====================================+==MLP FUNCTIONS=======================================
    def _initialize_parameters(self, n_features):
        """Initialize weights and biases for each layer using He initialization."""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) +[1]  # Output layer has 1 neuron for regression
        self.weights = []
        self.biases = []
        
        #adam
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i+1]
            # He initialization for ReLU
            if self.activation == 'relu':
                limit = np.sqrt(2.0 / fan_in)
                W = np.random.normal(0, limit, (fan_in, fan_out))
            else:
                # Xavier/Glorot for other activations
                limit = np.sqrt(6.0 / (fan_in + fan_out))
                W = np.random.uniform(-limit, limit, (fan_in, fan_out))
            b = np.zeros((1, fan_out))
            self.weights.append(W)
            self.biases.append(b)
            
            #adam
            self.m_weights.append(np.zeros((fan_in, fan_out)))
            self.v_weights.append(np.zeros((fan_in, fan_out)))
            self.m_biases.append(np.zeros((1, fan_out)))
            self.v_biases.append(np.zeros((1, fan_out)))
    
    def _forward_pass(self, X):
        """Perform forward propagation through the network."""
        activations = [X]
        zs = []  # Linear combinations before activation
        
        # Forward pass through hidden layers
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            zs.append(z)
            # Apply activation function
            if self.activation == 'relu':
                a = np.maximum(0, z)
            elif self.activation == 'tanh':
                a = np.tanh(z)
            elif self.activation == 'sigmoid':
                a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            else:
                raise ValueError(f"Unsupported activation function: {self.activation}")
            activations.append(a)
        
        # Output layer (no activation for regression)
        z_output = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        zs.append(z_output)
        y_pred = z_output  # Linear output for regression
        activations.append(y_pred)
        
        return activations, zs
    
    def _compute_loss(self, y, residuals):
        if self.loss_function is 'MSE': return np.mean((y - residuals) ** 2)
        elif self.loss_function is 'MAE': return np.mean(abs(y - residuals))
        elif self.loss_function is 'huber': return self._huber_loss(residuals)        
    
    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def _get_delta_update(self, delta, weights, z):
        if self.activation == 'relu': return np.dot(delta, weights.T) * self._relu_derivative(z)
        elif self.activation == 'tanh': return np.dot(delta, weights.T) * (z > 0).astype(float)
        elif self.activation == 'sigmoid':
            s = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            return np.dot(delta, weights.T) * s * (1-s)
            
    
    def _backward_pass(self, X, y, activations, zs, time=None):
        """Perform backpropagation to compute gradients."""
        m = X.shape[0]  # Number of samples
        deltas = []
        
        if self.optimiser is None:
            # Output layer delta (for regression, derivative of MSE w.r.t. z)
            delta = (activations[-1] - y) / m
            deltas.append(delta)

            # Backpropagate through hidden layers
            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)

            # Reverse deltas to match layer order
            deltas.reverse()
            self.delta_history.append(deltas)
            
            # Compute gradients for weights and biases
            for l in range(len(self.weights)):
                # Gradient of loss w.r.t. weights
                dW = np.dot(activations[l].T, deltas[l]) / m
                # Gradient of loss w.r.t. biases
                db = np.sum(deltas[l], axis=0, keepdims=True) / m
                # Update weights and biases
                self.weights[l] -= self.eta2 * dW 
                self.biases[l] -= self.eta2 * db
            
        elif self.optimiser is 'SGD':
            y_pred = activations[-1]

            delta = y_pred - y
            dW = np.dot(activations[-2].T, delta) / X.shape[0]
            db = np.sum(delta, axis=0) / X.shape[0]
            self.weights[-1] -= self.eta2 * dW
            self.biases[-1] -= self.eta2 * db

            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)
                self.delta_history.append(deltas)
                dW = np.dot(activations[l].T, delta) / X.shape[0]
                db = np.sum(delta, axis=0) / X.shape[0]

                self.weights[l] -= self.eta2 * dW
                self.biases[l] -= self.eta2 * db
                
            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])

        elif self.optimiser is 'adam':
            y_pred = activations[-1] # Output of the network

            # 2. backpropagation (using Adam slightly different)
            delta = y_pred - y
            dW = np.dot(activations[-2].T, delta) / X.shape[0] # Average over batch
            db = np.sum(delta, axis=0) / X.shape[0]

            # apply Adam updates to weights
            self.m_weights[-1] = self.beta2 * self.m_weights[-1] + (1 - self.beta2) * dW
            self.v_weights[-1] = self.beta3 * self.v_weights[-1] + (1 - self.beta3) * (dW ** 2)
            m_w_hat = self.m_weights[-1] / (1 - self.beta2**time)
            v_w_hat = self.v_weights[-1] / (1 - self.beta3**time)
            self.weights[-1] -= (self.eta2 * m_w_hat ) / (np.sqrt(v_w_hat) + self.epsilon2)

            # apply Adam updates to bias
            self.m_biases[-1] = self.beta2 * self.m_biases[-1] + (1 - self.beta2) * db
            self.v_biases[-1] = self.beta3 * self.v_biases[-1] + (1 - self.beta3) * (db ** 2)
            m_b_hat = self.m_biases[-1] / (1 - self.beta2**time)
            v_b_hat = self.v_biases[-1] / (1 - self.beta3**time)
            self.biases[-1] -= (self.eta2 * m_b_hat) / (np.sqrt(v_b_hat) + self.epsilon2)

            # Propagate gradients backward through hidden layers
            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)
                self.delta_history.append(deltas)
                dW_hidden = np.dot(activations[l].T, delta) / X.shape[0]
                db_hidden = np.sum(delta, axis=0) / X.shape[0]

                # apply Adam updates to weights
                self.m_weights[l] = self.beta2 * self.m_weights[l] + (1 - self.beta2) * dW_hidden
                self.v_weights[l] = self.beta3 * self.v_weights[l] + (1 - self.beta3) * (dW_hidden ** 2)
                m_w_hat = self.m_weights[l] / (1 - self.beta2**time)
                v_w_hat = self.v_weights[l] / (1 - self.beta3**time)
                self.weights[l] -= (self.eta2 * m_w_hat) / (np.sqrt(v_w_hat) + self.epsilon2)

                # apply Adam updates to bias
                self.m_biases[l] = self.beta2 * self.m_biases[l] + (1 - self.beta2) * db_hidden
                self.v_biases[l] = self.beta3 * self.v_biases[l] + (1 - self.beta3) * (db_hidden ** 2)
                m_b_hat = self.m_biases[l] / (1 - self.beta2**time)
                v_b_hat = self.v_biases[l] / (1 - self.beta3**time)
                self.biases[l] -= (self.eta2 * m_b_hat) / (np.sqrt(v_b_hat) + self.epsilon2)

            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])
            
        return self
        
    def _get_shuffled(self, n_samples, X, y):
        permutation = np.random.permutation(n_samples)
        X_shuffled = X[permutation]
        y_shuffled = y[permutation]
        return X_shuffled, y_shuffled
    
    def _set_epoch_loss(self, y, activation):
        epoch_loss = self._compute_loss(y, activation)
        self.loss_history.append(epoch_loss)
        return self
    
    def _store_preds(self, X, y, activations, t, epoch):
        preds = {}
        activations, _ = self._forward_pass(X)
        self._set_epoch_loss(y, activations[-1])
        preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
        self.estimator_preds.append(preds) # store the preds
        return self
    
    def _fit_MLP(self, t, X, residuals): #activations are multipled by learning rate and alpha
        X = np.asarray(X)
        y = np.asarray(residuals).reshape(-1, 1) #changing to y for easy coding
        n_samples, n_features = X.shape
        if self.batch_size is None: self.batch_size = n_samples #if no batch, set to number of samples
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        self.weights_history.append([w.copy() for w in self.weights])
        self.biases_history.append([b.copy() for b in self.biases])
        activations, _ = self._forward_pass(X)
        initial_loss = self._compute_loss(y, activations[-1])
        self.loss_history.append(initial_loss)
        
        # global time step for Adam bias correction
        time = 0
        
        # Training loop
        if self.optimiser is None:
            for epoch in range(self.n_epochs):
            # Shuffle data: -------------------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)
                # Mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i:i + self.batch_size]
                    y_batch = y_shuffled[i:i + self.batch_size]
                    # Forward pass
                    activations, zs = self._forward_pass(X_batch) #output for model prediction
                    # Backward pass
                    self._backward_pass(X_batch, y_batch, activations, zs)
                    self._store_preds(X_batch, y_batch, activations, t, epoch)
                #end loop---------------------------------------------------
        
        if self.optimiser is 'SGD':
            for epoch in range(self.n_epochs):
            # shuffle datasets
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)

                # mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]
                    #forward prop
                    activations, zs = self._forward_pass(X_batch)
                    #back prop
                    self._backward_pass(X_batch, y_batch, activations, zs)

                    #store results
                    self._store_preds(X_batch, y_batch, activations, t, epoch)

        if self.optimiser is 'adam':
             for epoch in range(self.n_epochs):
                # Shuffle data:  Gradient descent-----------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)
                
                #mini batch loop
                for i in range(0, n_samples, self.batch_size):
                    time += 1
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]

                    # 1. forward prop
                    activations, zs = self._forward_pass(X_batch)
                    # Backward prop
                    self._backward_pass(X_batch, y_batch, activations, zs, time)
                    #store results
                    self._store_preds(X_batch, y_batch, activations, t, epoch)

        return self
    
    #=======================================XGB FUNCTIONS=======================================
    def _xg_MSE(self, y, y_pred):
        gradient = 2 * (y_pred - y)
        hessian = np.full_like(y, 2)
        return gradient, hessian
    
    def _xg_friedman(self, y, y_pred):
        gradient = (y_pred - y)
        hessian = np.ones_like(y_pred)
        return gradient, hessian
    
    def _xg_MAE(self, y, y_pred):
        x = y_pred - y
        gradient = np.sign(x)
        gradient[np.abs(x) < self.tau] = 2/self.tau * x[np.abs(x) < self.tau] #smooth with tau near zero
        hessian = np.zeros_like(y)
        hessian[np.abs(x) < self.tau] = 2/self.tau
        return gradient, hessian
    
    def _xg_huber(self, y, y_pred):
        scale = 1 + ((y-y_pred)/ self.delta)**2 
        scale_sqrt = np.sqrt(scale)
        gradient = (y - y_pred) / scale_sqrt
        hessian = 1.0 / scale / scale_sqrt
        return gradient, hessian
    
    def _objective(self, y, y_pred):
        if self.objective is None: self.objective = self.loss_function
            
        if self.objective is 'MSE': gradients, hessians = self._xg_MSE(y, y_pred)
        elif self.objective is 'friedmanMSE': gradients, hessians = self._xg_friedman(y, y_pred)
        elif self.objective is 'MAE': gradients, hessians = self._xg_MAE(y, y_pred)
        elif self.objective is 'huber': gradients, hessians = self._xg_huber(y, y_pred)
        return gradients, hessians
    
    def _fit_xg(self, X, y):
        gradients, hessians = self._objective(y, self.current_pred)
        sample_idxs = None if self.subsample == 1.0 else self._get_subsample(y)
        booster = TreeBooster(X, gradients, hessians, self.min_child_weight, self.reg_gamma, self.reg_lambda, self.reg_alpha, self.colsample_bynode, self.max_depth, sample_idxs)    
        self.current_pred += self.learning_rate * booster.predict(X)
        self.models.append(booster)
        self.estimator_preds.append(self.current_pred)
        return self    
    
    #=======================================DECAY FUNCTIONS=======================================
    def _exponential_decay(self, t): return self.initial_learning_rate * (np.exp(-self.decay * t))
    
    def _expSchedule_decay(self, t): return self.decay * (self.initial_learning_rate ** t)#exponential decay from 0.1 to 0.01 over 100 rounds

    def _gradient_decay(self, gradients): return self.learning_rate / (np.sum(gradients**2))    

    def _beta_decay(self, t): return self.distribution[t]
    
    def _logistic_decay(self, t): return self.initial_learning_rate/(1+ np.exp(-self.decay * (t -(self.initial_learning_rate/2)))) #use high values of decay for a less than 0 eta
    
    def _linear_decay(self, t): return self.initial_learning_rate - (self.decay * t) #use low values of decay else whacky results ensue
    
    def _polynomial_decay(self, t): return 1 - (t / self.n_estimators)**self.power
    
    def _inverse_time_decay(self, t): return self.initial_learning_rate / (1 + self.decay * math.floor(t/self.decay_steps))

    def _step_decay(self, t): return self.initial_learning_rate * self.decay**(math.floor((1+t)/self.decay_steps)) #higher decay, smaller schedule
    
    def _adaptive_decay(self, gradients): #gradients are residuals
        self.G += gradients**2
        self.eta = self.initial_learning_rate / (np.sqrt(self.G + self.epsilon3)) * gradients  #need new epsilon
        return self.eta
    
    def _eta_decay(self, t, residuals):
        if self.decay_function is 'adaptive': self.eta = self._adaptive_decay(residuals)
        elif t is 0: self.learning_rate = self.learning_rate #skip decay on round 1
        elif self.decay_function is 'exponential': self.learning_rate = self._exponential_decay(t)
        elif self.decay_function is 'exponential2': self.learning_rate = self._expSchedule_decay(t)
        elif self.decay_function is 'gradient': self.learning_rate = self._gradient_decay(residuals)
        elif self.decay_function is 'beta': self.learning_rate = self._beta_decay(t)
        elif self.decay_function is 'logistic': self.learning_rate = self._logistic_decay(t)
        elif self.decay_function is 'linear': self.learning_rate = self._linear_decay(t)
        elif self.decay_function is 'polynomial': self.learning_rate = self._polynomial_decay(t)
        elif self.decay_function is 'inverse': self.learning_rate = self._inverse_time_decay(t)
        elif self.decay_function is 'step': self.learning_rate = self._step_decay(t)
            
        if self.learning_rate is 0: self.learning_rate += 0.00001  #to avoid zeroing out the learning. 
        return self
    
    def _get_best_distribution(self, data):
        dist_names = ["norm", "exponweib", "weibull_max", "weibull_min", "genextreme", "beta", "gamma", 'expon', 'cauchy', 'laplace']
        dist_results = []
        params = {}

        for dist_name in dist_names:
            dist = getattr(st, dist_name)
            param = dist.fit(data)
            params[dist_name] = param

            # Apply the Kolmogorov-Smirnov test
            D, p = st.kstest(data, dist_name, args=param)
#             print(f"p value for {dist_name} = {p}")
            dist_results.append((dist_name, p))
#         print(params)
        # Select the best fitting distribution based on the highest p-value
        best_dist, best_p = max(dist_results, key=lambda item: item)
        print(f"Best fitting distribution: {best_dist}")
#         print(f"Best p value: {best_p}")
#         print(f"Parameters for the best fit: {params[best_dist]}")

        return best_dist, best_p, params[best_dist]

    def _set_distribution(self, best_dist, params):
        distribution = getattr(st, best_dist)
        dist = distribution(*params)
        self.distribution = dist
        return self.distribution
    
    #=======================================Fit FUNCTIONS=======================================
    def _early_stopping(self, X, y, t):
        val_score = self.score(X, y)
        self.validation_scores.append(val_score)
#         print('Current score: ', val_score, '\nBest score: ', self.best_val_score, '\nTolerance: ', self.best_val_score - self.tolerance)
        if val_score < self.best_val_score - self.tolerance:
#             print('continue')
            self.best_val_score = val_score
            self.validation_scores.append(val_score)
            self.best_iteration = t + 1 #iterations are 0 inclusive, so need an extra 1
            self.no_improvement_count = 0 #reset the clock
        else: self.no_improvement_count += 1 #if going badly, lose patience
        
        self.n_estimators = self.best_iteration #update number of estimators so the prediction doesn't go awry

        if self.no_improvement_count >= self.patience:
            print(f'Early stopping at iteration {t+1}. Best validation {self.measure}: {self.best_val_score}')
            self.n_estimators = self.best_iteration #update number of estimators so the prediction doesn't go awry
            return True
            
        return False

    def _update_predictions(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.eta[i] * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.eta[i] * right_value) for i in range(len(X[:,feature_index]))])  
    
    def fit(self, X, y):
        if self.validation_fraction is not None: #start early stopping
            X, X_val, y, y_val = modsel.train_test_split(X, y, test_size = self.validation_fraction, random_state = self.seed)
        
        self.initial_pred = self._initialize_predictions(y)
        self.G = np.zeros(len(y)) #for adative decay
        
        if self.classifier is 'Xtreme': self.current_pred= np.full(len(y), self.base_prediction) #if not set, will default to 0
        else: self.current_pred = np.full(len(y), np.float64(self.initial_pred))
        #=============================fitting algorithm==========================================       
        for t in range(self.n_estimators): #growing t trees... ba-dump, tiss
            if t is 0: self.eta = np.full(len(y), self.initial_learning_rate)
            if self.decay_function is not 'adaptive':
                self.eta = np.full(len(y), self.learning_rate)
#             print('self.eta: ',self.eta)
            self.eta_history.append(self.eta) #storing for schedule analysis. it'll be a bit annoying to use later as array, but when fixed, can take any value
            #========================  
            residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
            #========================
#             for i in active_models:
            if self.classifier is 'stump': self._fit_stump(t, X, residuals)
            elif self.classifier is 'tree': self._fit_tree(t, X, residuals)
            elif self.classifier is 'MLP': self._fit_MLP(t, np.asarray(StandardScaler().fit_transform(X)), residuals)
            elif self.classifier is 'Xtreme': self._fit_xg(X, residuals)       
            
            #check early stopping criteria
            if self.validation_fraction is not None: 
#                 print(f'checking eraly stopping at round {t}')
                stop = self._early_stopping(X_val, y_val, t)
                if stop is True: break

              #now decay eta
            if self.decay_function is not None:
                self._eta_decay(t, residuals)
                
        #==============D.A.R.T start=============
#             if self.drop_rate is not None:
        self.active_models = list(range(len(self.models))) #DART for stumps and trees
#         print(f't: {t}, active_models {self.active_models}')
        if len(self.active_models) > 0 and self.drop_rate > 0:
            n_to_drop = int(len(self.active_models) * self.drop_rate)
            dropped_models = np.random.choice(self.active_models, n_to_drop, replace = False)
            self.active_models = [i for i in self.active_models if i not in dropped_models]
#             print(f't: {t}, active_models {self.active_models}')
            
            #======Do the D.A.R.T=====# reset residuals. get the indexes and remove the models from the list. then refit the model so far
            if self.classifier is 'Xtreme': self.DART_pred= np.full(len(y), self.base_prediction) #if not set, will default to 0
            else: self.DART_pred = np.full(len(y), np.float64(self.initial_pred))
                
            self.DART_models = [self.models[i] for i in self.active_models] #saved for prediction
            for i in self.active_models:
#                 residuals = self._compute_residuals(y, self.DART_pred)
                self.DART_pred += self.eta_history[i] * self.estimator_preds[i]
        #==============D.A.R.T end ==============           
            
        #find best distributions for eta if using adaptive
        if self.decay_function is 'adaptive':
            best_dist, best_p, params = self._get_best_distribution(self.eta_history[-1])
            self.distribution = self._set_distribution(best_dist, params)
            #now, when predicting, consult the dist and get etas for each value
        
        return self
    
    #=======================================PREDICT FUNCTIONS=======================================
    #self.eta_testing_history =, self.testing_preds
    def _predict_sample(self, x, node): #goes to bottom of tree to get prediction, will need to do update_predictions
        if node['is_leaf']: return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else: return self._predict_sample(x, node['right'])
    
    def _adaptive_update_preds(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.test_eta[i] * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.test_eta[i] * right_value) for i in range(len(X[:,feature_index]))])  

    def _predict_train(self, X): #for adative decay, this avoids doubleling the descent on training
        y_pred = np.full(len(X), self.initial_pred)
        if self.classifier is 'stump':
            if self.drop_rate > 0:

                for stump in self.DART_models:
                    y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X)
            else: 
                for stump in self.models:
                    y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
        
        elif self.classifier is 'tree': 
            if self.drop_rate > 0:
                print(self.active_models)
#                 print(len(self.DART_models), '\n')
                for t in range(len(self.DART_models)): #if doing early stopping, n_estimators won't work here, so need number of trees
                    predictions = [] 
                    current_tree = self.DART_models[t]

                    for x in X: predictions.append(self._predict_sample(x, current_tree))
                    y_pred = self._predict(predictions, y_pred, self.learning_rate)
            else:   
                for t in range(len(self.models)): #if doing early stopping, n_estimators won't work here, so need number of trees
                    predictions = [] #np.zeros(len(X))
                    current_tree = self.models[t]#[f'tree {t}']

                    for x in X: predictions.append(self._predict_sample(x, current_tree))
                    y_pred = self._predict(predictions, y_pred, self.learning_rate)
        
        elif self.classifier is 'MLP':
            X = np.asarray(StandardScaler().fit_transform(X))
            
            if self.drop_rate > 0:
                print(self.active_models)
                print(len(self.DART_models), '\n')
                for t in range(self.n_estimators):
                    activations, _ = self._forward_pass(X)
                    predictions = activations[-1].flatten()
                    y_pred = self._predict(predictions, y_pred, self.learning_rate)
            else:
                for t in range(self.n_estimators):
                    activations, _ = self._forward_pass(X)
                    predictions = activations[-1].flatten()
                    y_pred = self._predict(predictions, y_pred, self.learning_rate)
                
        elif self.classifier is 'Xtreme':
            if self.drop_rate > 0:
                return (self.base_prediction + self.learning_rate * np.sum([booster.predict(X) for booster in self.DART_models], axis=0))
            else: return (self.base_prediction + self.learning_rate * np.sum([booster.predict(X) for booster in self.models], axis=0))
        
        return y_pred
    
    def _predict(self, predictions, y_pred, eta):
        predictions = np.array(predictions) * eta  #self.learning_rate #will need to get indices and mask eta to work this out
        y_pred += predictions
        self.testing_preds.append(y_pred)
        return y_pred

    def predict(self, X): #if adaptive decay is being used, then calling predict on training data will fits it twice, pretty much
        y_pred = np.full(len(X), self.initial_pred)
        self.test_eta = np.full(len(X), self.initial_learning_rate)
#         pdf = self.distribution
            
        if self.classifier is 'stump':
            if self.drop_rate > 0:
                for stump in self.DART_models:
                    if self.decay_function is 'adaptive': #does adaptive decay on the stump
                        y_pred = self._adaptive_update_preds(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
                        self.testing_preds.append(y_pred)
                        self.test_eta = self.distribution.pdf(y_pred)
                    else: 
                        y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X)
                        self.testing_preds.append(y_pred)
                        
            else: #not DART
                for stump in self.models:
                    if self.decay_function is 'adaptive': #does adaptive decay on the stump
                        y_pred = self._adaptive_update_preds(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
                        self.testing_preds.append(y_pred)
                        self.test_eta = self.distribution.pdf(y_pred)  #need to use a different eta to avoid doubling the gradient descent
                    else: 
                        y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
                        self.testing_preds.append(y_pred)
          
        #iterate through each tree, get predictsion, weight them by learning rate and add the tree predictions to the total prediction (weight by gamme later)
        elif self.classifier is 'tree': 
            if self.drop_rate > 0:
                for t in range(len(self.DART_models)):
                    predictions = [] #np.zeros(len(X))
                    current_tree = self.DART_models[t]#[f'tree {t}']
                
                    for x in X: predictions.append(self._predict_sample(x, current_tree))
                
                    if self.decay_function is 'adaptive': 
                        y_pred =self._predict(predictions, y_pred, self.test_eta)
                        self.test_eta = self.distribution.pdf(y_pred)
                    
                    else: y_pred = self._predict(predictions, y_pred, self.learning_rate)
            else: #not DART 
                for t in range(len(self.models)):
                    predictions = [] #np.zeros(len(X))
                    current_tree = self.models[t]#[f'tree {t}']

                    for x in X: predictions.append(self._predict_sample(x, current_tree))

                    if self.decay_function is 'adaptive': 
                        y_pred =self._predict(predictions, y_pred, self.test_eta)
                        self.test_eta = self.distribution.pdf(y_pred)

                    else: y_pred = self._predict(predictions, y_pred, self.learning_rate)
        
        elif self.classifier is 'MLP':
            X = np.asarray(StandardScaler().fit_transform(X))
            if self.drop_rate > 0:
                for t in range(self.n_estimators):
                    activations, _ = self._forward_pass(X)
                    predictions = activations[-1].flatten()
                
                    if self.decay_function is 'adaptive':
                        y_pred =self._predict(predictions, y_pred, self.test_eta)
                        self.test_eta = self.distribution.pdf(y_pred)
                
                    else: y_pred = self._predict(predictions, y_pred, self.learning_rate)
            else: #not DART
                for t in range(self.n_estimators):
                    activations, _ = self._forward_pass(X)
                    predictions = activations[-1].flatten()

                    if self.decay_function is 'adaptive':
                        y_pred =self._predict(predictions, y_pred, self.test_eta)
                        self.test_eta = self.distribution.pdf(y_pred)

                    else: y_pred = self._predict(predictions, y_pred, self.learning_rate)
                    
        elif self.classifier is 'Xtreme':
            if self.drop_rate > 0:
                for t in range(len(self.DART_models)):
                    predictions = self.DART_models[t].predict(X)
                
                    if self.decay_function is 'adaptive':
                        y_pred =self._predict(predictions, y_pred, self.test_eta)
                        self.test_eta = self.distribution.pdf(y_pred)
                    else: 
                        y_pred = self._predict(predictions, y_pred, self.learning_rate)
            else: #not dart
                for t in range(self.n_estimators):
                    predictions = self.models[t].predict(X)

                    if self.decay_function is 'adaptive':
                        y_pred =self._predict(predictions, y_pred, self.test_eta)
                        self.test_eta = self.distribution.pdf(y_pred)
                    else: 
                        y_pred = self._predict(predictions, y_pred, self.learning_rate)
        
        return y_pred   
    
    #=======================================SCORE FUNCTION=======================================
    def score(self, X, y): #for grid search and CV etc
        y_pred = self.predict(X)
        if self.measure is 'MSE': return np.mean((y - y_pred)**2)
        elif self.measure is 'RMSE': return np.sqrt(np.mean((y - y_pred) ** 2))
        elif self.measure is 'MAE': return np.mean(np.abs(y - y_pred))
        elif self.measure is 'MAPE': return np.mean(np.abs((y - y_pred) / y))
        elif self.measure is 'R2': return 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))
        return 

In [ ]:
class MLPRegressor(BaseEstimator, RegressorMixin):
    """
    A custom implementation of a Multi-Layer Perceptron Regressor with forward and backward propagation.
    Uses gradient descent for optimization.
    """
    
    def __init__(self, hidden_layer_sizes=(10,), activation='relu',optimiser ='SGD',beta2 =0.9, beta3 = 0.999, eta = 1e-8, alpha=0.01, 
                 n_epochs=1000, batch_size=32, random_state=None):
        """
        Initialize the MLP Regressor.

        Parameters:
        - hidden_layer_sizes: tuple, number of neurons in each hidden layer
        - activation: str, activation function for hidden layers ('relu', 'tanh', 'sigmoid')
        - learning_rate: float, step size for gradient descent
        - n_epochs: int, number of training iterations
        - batch_size: int, size of mini-batches
        - random_state: int, seed for reproducibility
        """
        self.hidden_layer_sizes = hidden_layer_sizes
        self.optimiser = optimiser
        self.activation = activation
        self.alpha = alpha
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.weights = []
        self.biases = []
        self.loss_history = []
        #adam
        self.beta2 = beta2
        self.beta3 = beta3
        self.eta = eta
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        self.weights_history = []
        self.biases_history = []
        
        # Set random seed if provided
        if self.seed:
            np.random.seed(self.seed)
    
    def _initialize_parameters(self, n_features):
        """Initialize weights and biases for each layer using He initialization."""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) +[1]  # Output layer has 1 neuron for regression
        self.weights = []
        self.biases = []
        
        #adam
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i+1]
            # He initialization for ReLU
            if self.activation == 'relu':
                limit = np.sqrt(2.0 / fan_in)
                W = np.random.normal(0, limit, (fan_in, fan_out))
            else:
                # Xavier/Glorot for other activations
                limit = np.sqrt(6.0 / (fan_in + fan_out))
                W = np.random.uniform(-limit, limit, (fan_in, fan_out))
            b = np.zeros((1, fan_out))
            self.weights.append(W)
            self.biases.append(b)
            
            #adam
            self.m_weights.append(np.zeros((fan_in, fan_out)))
            self.v_weights.append(np.zeros((fan_in, fan_out)))
            self.m_biases.append(np.zeros((1, fan_out)))
            self.v_biases.append(np.zeros((1, fan_out)))
    
    def _forward_pass(self, X):
        """Perform forward propagation through the network."""
        activations = [X]
        zs = []  # Linear combinations before activation
        
        # Forward pass through hidden layers
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            zs.append(z)
            # Apply activation function
            if self.activation == 'relu':
                a = np.maximum(0, z)
            elif self.activation == 'tanh':
                a = np.tanh(z)
            elif self.activation == 'sigmoid':
                a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            else:
                raise ValueError(f"Unsupported activation function: {self.activation}")
            activations.append(a)
        
        # Output layer (no activation for regression)
        z_output = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        zs.append(z_output)
        y_pred = z_output  # Linear output for regression
        activations.append(y_pred)
        
        return activations, zs
    
    def _compute_loss(self, y_true, y_pred):
        """Compute mean squared error loss.""" #put loss in here, then check if loss is better or not
        return np.mean((y_true - y_pred) ** 2)
    
    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def _backward_pass(self, X, y_true, activations, zs):
        """Perform backpropagation to compute gradients."""
        m = X.shape[0]  # Number of samples
        deltas = []
        
        # Output layer delta (for regression, derivative of MSE w.r.t. z)
        delta_output = (activations[-1] - y_true) / m
        deltas.append(delta_output)
        
        # Backpropagate through hidden layers
        for l in range(len(self.weights) - 2, -1, -1):
            # Delta for current layer
            delta = np.dot(deltas[-1], self.weights[l+1].T)
            # Apply derivative of activation function
            if self.activation == 'relu':
                delta *= (zs[l] > 0).astype(float)
            elif self.activation == 'tanh':
                delta *= (1 - np.tanh(zs[l]) ** 2)
            elif self.activation == 'sigmoid':
                s = 1 / (1 + np.exp(-np.clip(zs[l], -500, 500)))
                delta *= s * (1 - s)
            deltas.append(delta)
        
        # Reverse deltas to match layer order
        deltas.reverse()
        
        # Compute gradients for weights and biases
        for l in range(len(self.weights)):
            # Gradient of loss w.r.t. weights
            dW = np.dot(activations[l].T, deltas[l]) / m
            # Gradient of loss w.r.t. biases
            db = np.sum(deltas[l], axis=0, keepdims=True) / m
            # Update weights and biases
            self.weights[l] -= self.alpha * dW
            self.biases[l] -= self.alpha * db
    
    def fit(self, X, y):
        """
        Train the MLP regressor using gradient descent with mini-batches.
        Parameters:
        - X: array-like, shape (n_samples, n_features)
        - y: array-like, shape (n_samples,)
        Returns:
        - self
        """
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        #adam
        self.weights_history.append([w.copy() for w in self.weights])
        self.biases_history.append([b.copy() for b in self.biases])
        activations, _ = self._forward_pass(X)
        initial_loss = self._compute_loss(y, activations[-1])
        self.loss_history.append(initial_loss)
        # global time step for Adam bias correction
        t = 0
        
        # Training loop
        if self.optimiser is 'SGD':
            for epoch in range(self.n_epochs):
                # Shuffle data: Stochastic Gradient descent-----------------------------------
                permutation = np.random.permutation(n_samples)
                X_shuffled = X[permutation]
                y_shuffled = y[permutation]

                # Mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i:i + self.batch_size]
                    y_batch = y_shuffled[i:i + self.batch_size]

                    # Forward pass
                    activations, zs = self._forward_pass(X_batch) #output for model prediction

                    # Compute loss
                    loss = self._compute_loss(y_batch, activations[-1])

                    # Backward pass
                    self._backward_pass(X_batch, y_batch, activations, zs)
                #Stochastic Gradient descent end----------------------------------------------
                # Record loss every 100 epochs
                if (epoch + 1) % 100 == 0:
                    activations, _ = self._forward_pass(X)
                    epoch_loss = self._compute_loss(y, activations[-1])
                    self.loss_history.append(epoch_loss)
                    print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss:.6f}")
        
        if self.optimiser is 'adam':
             for epoch in range(self.n_epochs):
                # Shuffle data: Stochastic Gradient descent-----------------------------------
                permutation = np.random.permutation(n_samples)
                X_shuffled = X[permutation]
                y_shuffled = y[permutation]
                
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]

                    t += 1

                    # 1. forward pass
                    activations, zs = self._forward_pass(X_batch)
                    y_pred = activations[-1] # Output of the network

                    # 2. backpropagation (using Adam slightly different than SGD)
                    delta = y_pred - y_batch
                    grad_w_output = np.dot(activations[-2].T, delta) / X_batch.shape[0] # Average over batch
                    grad_b_output = np.sum(delta, axis=0) / X_batch.shape[0]

                    # apply Adam updates to weights
                    self.m_weights[-1] = self.beta2 * self.m_weights[-1] + (1 - self.beta2) * grad_w_output
                    self.v_weights[-1] = self.beta3 * self.v_weights[-1] + (1 - self.beta3) * (grad_w_output ** 2)
                    m_w_hat = self.m_weights[-1] / (1 - self.beta2**t)
                    v_w_hat = self.v_weights[-1] / (1 - self.beta3**t)
                    self.weights[-1] -= self.alpha * m_w_hat / (np.sqrt(v_w_hat) + self.eta)

                    # apply Adam updates to bias
                    self.m_biases[-1] = self.beta2 * self.m_biases[-1] + (1 - self.beta2) * grad_b_output
                    self.v_biases[-1] = self.beta3 * self.v_biases[-1] + (1 - self.beta3) * (grad_b_output ** 2)
                    m_b_hat = self.m_biases[-1] / (1 - self.beta2**t)
                    v_b_hat = self.v_biases[-1] / (1 - self.beta3**t)
                    self.biases[-1] -= self.alpha * m_b_hat / (np.sqrt(v_b_hat) + self.eta)


                    # Propagate gradients backward through hidden layers
                    for l in range(len(self.weights) - 2, -1, -1):
                        delta = np.dot(delta, self.weights[l+1].T) * self._relu_derivative(zs[l]) # d_activation(z)
                        grad_w_hidden = np.dot(activations[l].T, delta) / X_batch.shape[0]
                        grad_b_hidden = np.sum(delta, axis=0) / X_batch.shape[0]

                        # apply Adam updates to weights
                        self.m_weights[l] = self.beta2 * self.m_weights[l] + (1 - self.beta2) * grad_w_hidden
                        self.v_weights[l] = self.beta3 * self.v_weights[l] + (1 - self.beta3) * (grad_w_hidden ** 2)
                        m_w_hat = self.m_weights[l] / (1 - self.beta2**t)
                        v_w_hat = self.v_weights[l] / (1 - self.beta3**t)
                        self.weights[l] -= self.alpha * m_w_hat / (np.sqrt(v_w_hat) + self.eta)

                        # apply Adam updates to bias
                        self.m_biases[l] = self.beta2 * self.m_biases[l] + (1 - self.beta2) * grad_b_hidden
                        self.v_biases[l] = self.beta3 * self.v_biases[l] + (1 - self.beta3) * (grad_b_hidden ** 2)
                        m_b_hat = self.m_biases[l] / (1 - self.beta2**t)
                        v_b_hat = self.v_biases[l] / (1 - self.beta3**t)
                        self.biases[l] -= self.alpha * m_b_hat / (np.sqrt(v_b_hat) + self.eta)


                self.weights_history.append([w.copy() for w in self.weights])
                self.biases_history.append([b.copy() for b in self.biases])

                activations, _ = self._forward_pass(X)
                epoch_loss = self._compute_loss(y, activations[-1])
                self.loss_history.append(epoch_loss)

                if (epoch + 1) % 100 == 0:
                    print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss:.4f}")
                
        return self
    
    def predict(self, X):
        """
        Predict using the trained model.
        Parameters:
        - X: array-like, shape (n_samples, n_features)
        Returns:
        - y_pred: array, shape (n_samples,)
        """
        X = np.asarray(X)
        activations, _ = self._forward_pass(X)
        return activations[-1].flatten()

In [ ]:
%autoreload
class GBoost(BaseEstimator, RegressorMixin):
    def __init__(self, classifier = 'stump', n_estimators = 10, learning_rate = 0.5, max_depth = 1, 
                 subsample = None, epsilon = 0.5, beta = 0.5, min_samples_leaf = 0, min_samples_split = 0, 
                 loss_function = 'MSE', hidden_layer_sizes=(10,), activation='relu', optimiser ='SGD',
                 beta2 =0.9, beta3 = 0.999, eta = 1e-8, alpha=0.01, n_epochs=1000, batch_size=32,
                 seed = 21, score = 'MSE'):
        
        self.classifier = classifier
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.epsilon = epsilon #for huber loss
        self.beta = beta #for huber loss
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.subsample = subsample
        self.loss_function = loss_function
        self.seed = seed
        self.score = score
        self.stumps = []
        self.loss_history = []  #records loss for plotting
        self.initial_pred = None
        self.current_pred = None
        self.root = None
        self.trees = []
        
        #MLP params
        self.hidden_layer_sizes = hidden_layer_sizes
        self.optimiser = optimiser
        self.activation = activation
        self.alpha = alpha
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.weights = []
        self.biases = []
        self.estimator_preds = []
        #adam
        self.beta2 = beta2
        self.beta3 = beta3
        self.eta = eta
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        self.weights_history = []
        self.biases_history = []
        
        # Set random seed if provided
        if self.seed:
            np.random.seed(self.seed)
        
    def _compute_residuals(self, y, y_pred): 
        if self.loss_function is 'huber': return self._huber_gradient((y - y_pred))
        else: return y - y_pred
        
    def _get_subsample(self, y): #return indices of sub sampled data
        np.random.seed(self.seed)
        return np.array(np.random.choice(len(y), size=(round(len(y) * self.subsample)), replace=False))
    
    def _huber_loss(self, residuals): #to find leaf values with huber loss minimisation
        abs_resid = np.abs(residuals)
        quadratic = self.beta * residuals**2
        linear = self.epsilon * (abs_resid - 0.5 * self.epsilon)

        return np.where(abs_resid <= self.epsilon, quadratic, linear).sum()
    
    def _initialize_predictions(self, y): #also use for leaf value
        if self.loss_function is 'MAE': return np.median(y)
        elif self.loss_function is 'huber': return self._huber_initialise_predictions(y)
        else: return y.mean()   
    
    def _huber_initialise_predictions(self, y): #to start the preds
        best_loss = float('inf')
        ip = None
        for i in y:
            y_t = self._huber_loss(i) #loss at this point
            total_loss = self._huber_loss(y - y_t) #loss across all data using that point
            if total_loss < best_loss: #minimise loss over data
                best_loss = total_loss
                ip = i                 #get minimising initial prediction         
        return ip
    
    def _huber_gradient(self, residuals): #transforms residuals inot a huber gradient. 
        abs_resid = np.abs(residuals)
        gradient = np.where(abs_resid <= self.epsilon, -residuals, -self.epsilon * np.sign(residuals))
        return gradient
    
    def _mse_(self, residuals, left_mask, right_mask):
        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        mse = (np.mean((residuals[left_mask] - left_value) ** 2) * np.sum(left_mask) + 
           np.mean((residuals[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
        return mse, left_value, right_value

    def _abs_error(self, residuals, left_mask, right_mask): #note, residuals have to be median, not mean for abs_error: outliers
        sign_vector = np.sign(residuals)

        left_value = sign_vector[left_mask].mean()
        right_value = sign_vector[right_mask].mean()

        mae = (np.mean((sign_vector[left_mask] - left_value) ** 2) * np.sum(left_mask) +
               np.mean((sign_vector[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)

        return mae, left_value, right_value

    def _friedman_MSE(self, residuals, left_mask, right_mask, weights = None):
        if weights is None:
            weights = np.ones(len(residuals))

        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        w_l = np.sum(weights[left_mask]) 
        w_r = np.sum(weights[right_mask])

        mean_l = np.mean(residuals[left_mask])
        mean_r = np.mean(residuals[right_mask])

        friedman_mse = ((w_l * w_r) / (w_l + w_r)) * ((mean_l - mean_r)**2)

        return friedman_mse, left_value, right_value

    def _huber(self, residuals, left_mask, right_mask):
        huber_residuals = self._huber_gradient(residuals) #always huber-ise the residuals, even when already hubered from r-1 

        left_value = np.mean(huber_residuals[left_mask]) if np.any(left_mask) else 0
        right_value = np.mean(huber_residuals[right_mask]) if np.any(right_mask) else 0
        #need to figure out best huber loss over all predictions. 
        y_pred = np.full_like(huber_residuals, left_value)
        y_pred[right_mask] = right_value
        
        error = self._huber_loss(self._compute_residuals(huber_residuals, y_pred))
        
        return error, left_value, right_value    

    def _train_weak_learner(self, X, residuals):
        X = np.array(X)  # Ensure X is a NumPy array
        n_samples, n_features = X.shape
        best_feature_index = None
        best_threshold = None
        best_left_value = None
        best_right_value = None

        if self.loss_function is 'friedmanMSE': best_error = float('-inf')
        else: best_error = float('inf')

        # Iterate over each feature
        for feature_index in range(n_features):
            thresholds = np.unique(X[:, feature_index])

            # Test each threshold as a potential split point
            for threshold in thresholds:
                left_mask = X[:, feature_index] <= threshold
                right_mask = X[:, feature_index] > threshold

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                    
                #skip if either side has fewer samples than the specified minimum samples per leaf
                if (np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf): continue
                    
                #impurity calculation
                if self.loss_function is 'MSE': error, left_value, right_value =  self._mse_(residuals, left_mask, right_mask)
                elif self.loss_function is 'friedmanMSE': error, left_value, right_value =  self._friedman_MSE(residuals, left_mask, right_mask)
                elif self.loss_function is 'MAE': error, left_value, right_value =  self._abs_error(residuals, left_mask, right_mask)
                elif self.loss_function is 'huber': error, left_value, right_value = self._huber(residuals, left_mask, right_mask)

                    
    #             If this is the best split so far, store it. FreidmanMSE maximises, so we need a different approach
                if self.loss_function is 'friedmanMSE':
                    if error > best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value
                else:
                    if error < best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value

        return best_feature_index, best_threshold, best_left_value, best_right_value, best_error
    
    def _fit_stump(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            stump = self._train_weak_learner(X[indices], resid_subsample)
            feature_index, threshold, left_value, right_value, best_error = stump

            current_pred_subsample = self.current_pred[indices]
            updates = self._update_predictions(current_pred_subsample, feature_index, threshold, left_value, right_value, X[indices])
            self.current_pred[indices] = updates
            #========================
        else: 
            stump = self._train_weak_learner(X, residuals)
            feature_index, threshold, left_value, right_value, best_error = stump
            self.current_pred = self._update_predictions(self.current_pred, feature_index, threshold, left_value, right_value, X)
        #========================
        self.stumps.append(stump)
        self.loss_history.append(best_error)
        return self
    
    #=======================================TREE FUNCTIONS=======================================
    def _leaf_value(self, y): return np.mean(y) #y is residuals for GBoost, can use initialise because the prediction is the mean
    
    def _grow_tree(self, X, residuals, depth = 0): #recursively grow the decision tree up to the specified depth. Which is a stump if not specified
        n_samples, n_features = X.shape
        #n_classes = len(np.unique(y)) #use this to stop if there are too few values
        
        #check stopping criteria
        if (depth >= self.max_depth or n_samples < self.min_samples_split): # or n_classes == 1):
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        #find best split
        best_feature, best_threshold, left_value, right_value, best_error = self._train_weak_learner(X, residuals)
        
        #if no split found, create a leaf
        if best_feature is None: 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        self.current_pred = self._update_predictions(self.current_pred, best_feature, best_threshold, left_value, right_value, X_train)
        
        #split data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold
        left_subtree = self._grow_tree(X[left_mask], residuals[left_mask], depth +1)
        right_subtree = self._grow_tree(X[right_mask], residuals[right_mask], depth +1)
                
        #track the tree stats
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'left value': left_value,
            'right value': right_value,
            'left': left_subtree,
            'right': right_subtree,
            'error': best_error,
            'is_leaf': False
        }
    
    def _fit_tree(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            self.root = self._grow_tree(X[indices], resid_subsample)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.error.loc[t,'error'] = self.root['error']
            self.trees.append(tree)
            #========================
        else:
            self.root = self._grow_tree(X, residuals)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.loss_history.append(best_error)
            self.trees.append(tree)
        return self
            
    #=====================================+==MLP FUNCTIONS=======================================
    def _initialize_parameters(self, n_features):
        """Initialize weights and biases for each layer using He initialization."""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) +[1]  # Output layer has 1 neuron for regression
        self.weights = []
        self.biases = []
        
        #adam
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i+1]
            # He initialization for ReLU
            if self.activation == 'relu':
                limit = np.sqrt(2.0 / fan_in)
                W = np.random.normal(0, limit, (fan_in, fan_out))
            else:
                # Xavier/Glorot for other activations
                limit = np.sqrt(6.0 / (fan_in + fan_out))
                W = np.random.uniform(-limit, limit, (fan_in, fan_out))
            b = np.zeros((1, fan_out))
            self.weights.append(W)
            self.biases.append(b)
            
            #adam
            self.m_weights.append(np.zeros((fan_in, fan_out)))
            self.v_weights.append(np.zeros((fan_in, fan_out)))
            self.m_biases.append(np.zeros((1, fan_out)))
            self.v_biases.append(np.zeros((1, fan_out)))
    
    def _forward_pass(self, X):
        """Perform forward propagation through the network."""
        activations = [X]
        zs = []  # Linear combinations before activation
        
        # Forward pass through hidden layers
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            zs.append(z)
            # Apply activation function
            if self.activation == 'relu':
                a = np.maximum(0, z)
            elif self.activation == 'tanh':
                a = np.tanh(z)
            elif self.activation == 'sigmoid':
                a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            else:
                raise ValueError(f"Unsupported activation function: {self.activation}")
            activations.append(a)
        
        # Output layer (no activation for regression)
        z_output = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        zs.append(z_output)
        y_pred = z_output  # Linear output for regression
        activations.append(y_pred)
        
        return activations, zs
    
    def _compute_loss(self, y, residuals):
        """Compute mean squared error loss.""" #put loss in here, then check if loss is better or not
        if self.loss_function is 'MSE': return np.mean((y - residuals) ** 2)
        elif self.loss_function is 'MAE': return np.mean(abs(y - residuals))
        elif self.loss_function is 'huber': return _huber_loss(residuals)        
    
    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def _backward_pass(self, X, y, activations, zs):
        """Perform backpropagation to compute gradients."""
        m = X.shape[0]  # Number of samples
        deltas = []
        
        # Output layer delta (for regression, derivative of MSE w.r.t. z)
        delta_output = (activations[-1] - y) / m
        deltas.append(delta_output)
        
        # Backpropagate through hidden layers
        for l in range(len(self.weights) - 2, -1, -1):
            # Delta for current layer
            delta = np.dot(deltas[-1], self.weights[l+1].T)
            # Apply derivative of activation function
            if self.activation == 'relu':
                delta *= (zs[l] > 0).astype(float)
            elif self.activation == 'tanh':
                delta *= (1 - np.tanh(zs[l]) ** 2)
            elif self.activation == 'sigmoid':
                s = 1 / (1 + np.exp(-np.clip(zs[l], -500, 500)))
                delta *= s * (1 - s)
            deltas.append(delta)
        
        # Reverse deltas to match layer order
        deltas.reverse()
        
        # Compute gradients for weights and biases
        for l in range(len(self.weights)):
            # Gradient of loss w.r.t. weights
            dW = np.dot(activations[l].T, deltas[l]) / m
            # Gradient of loss w.r.t. biases
            db = np.sum(deltas[l], axis=0, keepdims=True) / m
            # Update weights and biases
            self.weights[l] -= self.alpha * dW # * self.learning_rate
            self.biases[l] -= self.alpha * db # * self.learning_rate
    
    def _SGD():
        return
    
    def _adam():
        return
    
    def _fit_MLP(self, t, X, residuals): #activations are multipled by learning rate and alpha
        preds = {}
        X = np.asarray(X)
        y = np.asarray(residuals).reshape(-1, 1) #changing to y for easy coding
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        #adam
        self.weights_history.append([w.copy() for w in self.weights])
        self.biases_history.append([b.copy() for b in self.biases])
        activations, _ = self._forward_pass(X)
        initial_loss = self._compute_loss(y, activations[-1])
        self.loss_history.append(initial_loss)
        
        # global time step for Adam bias correction
        time = 0
        
        # Training loop
        if self.optimiser is 'SGD':
            for epoch in range(self.n_epochs):
                # Shuffle data: Stochastic Gradient descent-----------------------------------
                permutation = np.random.permutation(n_samples)
                X_shuffled = X[permutation]
                y_shuffled = y[permutation]

                # Mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i:i + self.batch_size]
                    y_batch = y_shuffled[i:i + self.batch_size]

                    # Forward pass
                    activations, zs = self._forward_pass(X_batch) #output for model prediction

                    # Compute loss
                    loss = self._compute_loss(y_batch, activations[-1])

                    # Backward pass
                    self._backward_pass(X_batch, y_batch, activations, zs)
                #Stochastic Gradient descent end----------------------------------------------
                activations, _ = self._forward_pass(X)
                preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
                self.estimator_preds.append(preds) # store the preds

                # Record loss every 100 epochs
                if (epoch + 1) % 100 == 0:
                    epoch_loss = self._compute_loss(y, activations[-1])
                    self.loss_history.append(epoch_loss)
    #               print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss:.6f}")
        
        if self.optimiser is 'adam':
             for epoch in range(self.n_epochs):
                # Shuffle data: Stochastic Gradient descent-----------------------------------
                permutation = np.random.permutation(n_samples)
                X_shuffled = X[permutation]
                y_shuffled = y[permutation]
                
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]

                    time += 1

                    # 1. forward pass
                    activations, zs = self._forward_pass(X_batch)
                    y_pred = activations[-1] # Output of the network

                    # 2. backpropagation (using Adam slightly different than SGD)
                    delta = y_pred - y_batch
                    grad_w_output = np.dot(activations[-2].T, delta) / X_batch.shape[0] # Average over batch
                    grad_b_output = np.sum(delta, axis=0) / X_batch.shape[0]

                    # apply Adam updates to weights
                    self.m_weights[-1] = self.beta2 * self.m_weights[-1] + (1 - self.beta2) * grad_w_output
                    self.v_weights[-1] = self.beta3 * self.v_weights[-1] + (1 - self.beta3) * (grad_w_output ** 2)
                    m_w_hat = self.m_weights[-1] / (1 - self.beta2**t)
                    v_w_hat = self.v_weights[-1] / (1 - self.beta3**t)
                    self.weights[-1] -= (self.alpha * m_w_hat ) / (np.sqrt(v_w_hat) + self.eta)

                    # apply Adam updates to bias
                    self.m_biases[-1] = self.beta2 * self.m_biases[-1] + (1 - self.beta2) * grad_b_output
                    self.v_biases[-1] = self.beta3 * self.v_biases[-1] + (1 - self.beta3) * (grad_b_output ** 2)
                    m_b_hat = self.m_biases[-1] / (1 - self.beta2**t)
                    v_b_hat = self.v_biases[-1] / (1 - self.beta3**t)
                    self.biases[-1] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

                    # Propagate gradients backward through hidden layers
                    for l in range(len(self.weights) - 2, -1, -1):
                        delta = np.dot(delta, self.weights[l+1].T) * self._relu_derivative(zs[l]) # d_activation(z)
                        grad_w_hidden = np.dot(activations[l].T, delta) / X_batch.shape[0]
                        grad_b_hidden = np.sum(delta, axis=0) / X_batch.shape[0]

                        # apply Adam updates to weights
                        self.m_weights[l] = self.beta2 * self.m_weights[l] + (1 - self.beta2) * grad_w_hidden
                        self.v_weights[l] = self.beta3 * self.v_weights[l] + (1 - self.beta3) * (grad_w_hidden ** 2)
                        m_w_hat = self.m_weights[l] / (1 - self.beta2**t)
                        v_w_hat = self.v_weights[l] / (1 - self.beta3**t)
                        self.weights[l] -= (self.alpha * m_w_hat) / (np.sqrt(v_w_hat) + self.eta)

                        # apply Adam updates to bias
                        self.m_biases[l] = self.beta2 * self.m_biases[l] + (1 - self.beta2) * grad_b_hidden
                        self.v_biases[l] = self.beta3 * self.v_biases[l] + (1 - self.beta3) * (grad_b_hidden ** 2)
                        m_b_hat = self.m_biases[l] / (1 - self.beta2**t)
                        v_b_hat = self.v_biases[l] / (1 - self.beta3**t)
                        self.biases[l] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

                    self.weights_history.append([w.copy() for w in self.weights])
                    self.biases_history.append([b.copy() for b in self.biases])

                    activations, _ = self._forward_pass(X)
                    epoch_loss = self._compute_loss(y, activations[-1])
                    self.loss_history.append(epoch_loss)

                    preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
                    self.estimator_preds.append(preds) # store the preds

#                 if (epoch + 1) % 100 == 0:
#                     print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss:.4f}")
        return self
    #=======================================Fit FUNCTIONS=======================================

    def _update_predictions(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.learning_rate * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.learning_rate * right_value) for i in range(len(X[:,feature_index]))])  
    
    def fit(self, X, y):
        self.initial_pred = self._initialize_predictions(y)
        self.current_pred = np.full(len(y), self.initial_pred)
                
        if self.classifier is 'stump':
            for t in range(self.n_estimators): 
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================  
                self._fit_stump(t, X, residuals)
                
        elif self.classifier is 'tree':
            for t in range(self.n_estimators): #growing t trees... ba-dump, tiss
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================
                self._fit_tree(t, X, residuals)
            
        elif self.classifier is 'MLP':
            X_test_scaled = scaler.fit_transform(X_test)
            
            for t in range(self.n_estimators):
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================
                self._fit_MLP(t, X, residuals)
                
        return self
    
    def _predict_sample(self, x, node): #goes to bottom of tree to get prediction, will need to do update_predictions
        if node['is_leaf']: return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else: return self._predict_sample(x, node['right'])
    
    def predict(self, X):
        y_pred = np.full(len(X), self.initial_pred)
        
        if self.classifier is 'stump':
            for stump in self.stumps:
                y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
            return y_pred
          
        #iterate through each tree, get predictsion, weight them by learning rate and add the tree predictions to the total prediction (weight by gamme later)
        elif self.classifier is 'tree': 
            for t in range(self.n_estimators):
                predictions = [] #np.zeros(len(X))
                current_tree = self.trees[t][f'tree {t}']
                
                for x in X:
                    predictions.append(self._predict_sample(x, current_tree))
                predictions = np.array(predictions) * self.learning_rate
                y_pred += predictions
        
        elif self.classifier is 'MLP':
            X = StandardScaler().fit_transform(X)
#             print(self.activation_preds)
            for t in range(self.n_estimators):
                X = np.asarray(X)
                activations, _ = self._forward_pass(X)
                predictions = activations[-1].flatten()
                predictions *= self.learning_rate #updated during forward and back prop
                y_pred += predictions #add to y_pred
          
        return y_pred   
        
    def score(self, X, y): #for grid search and CV etc
        y_pred = self.predict(X)
        if self.score is 'MSE': return np.mean((y - y_pred)**2)
        elif self.score is 'RMSE': return np.sqrt(np.mean((y - y_pred) ** 2))
        elif self.score is 'MAE': return np.mean(np.abs(y - y_pred))
        elif self.score is 'MAPE': return np.mean(np.abs((y - y_pred) / y))
        elif self.score is 'R2': return 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))
        return 

In [ ]:
%autoreload
class GBoost(BaseEstimator, RegressorMixin):
    def __init__(self, classifier = 'stump', n_estimators = 10, learning_rate = 0.5, max_depth = 1, 
                 subsample = None, epsilon = 0.5, beta = 0.5, min_samples_leaf = 0, min_samples_split = 0, 
                 loss_function = 'MSE', hidden_layer_sizes=(10,), activation='relu', optimiser ='SGD',
                 beta2 =0.9, beta3 = 0.999, eta = 1e-8, alpha=0.01, n_epochs=1000, batch_size=32,
                 seed = 21, score = 'MSE'):
        
        self.classifier = classifier
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.epsilon = epsilon #for huber loss
        self.beta = beta #for huber loss
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.subsample = subsample
        self.loss_function = loss_function
        self.seed = seed
        self.score = score
        self.stumps = []
        self.loss_history = []  #records loss for plotting
        self.initial_pred = None
        self.current_pred = None
        self.root = None
        self.trees = []
        
        #MLP params
        self.hidden_layer_sizes = hidden_layer_sizes
        self.optimiser = optimiser
        self.activation = activation
        self.alpha = alpha
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.weights = []
        self.biases = []
        self.estimator_preds = []
        #adam
        self.beta2 = beta2
        self.beta3 = beta3
        self.eta = eta
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        self.weights_history = []
        self.biases_history = []
        
        # Set random seed if provided
        if self.seed:
            np.random.seed(self.seed)
        
    def _compute_residuals(self, y, y_pred): 
        if self.loss_function is 'huber': return self._huber_gradient((y - y_pred))
        else: return y - y_pred
        
    def _get_subsample(self, y): #return indices of sub sampled data
        np.random.seed(self.seed)
        return np.array(np.random.choice(len(y), size=(round(len(y) * self.subsample)), replace=False))
    
    def _huber_loss(self, residuals): #to find leaf values with huber loss minimisation
        abs_resid = np.abs(residuals)
        quadratic = self.beta * residuals**2
        linear = self.epsilon * (abs_resid - 0.5 * self.epsilon)

        return np.where(abs_resid <= self.epsilon, quadratic, linear).sum()
    
    def _initialize_predictions(self, y): #also use for leaf value
        if self.loss_function is 'MAE': return np.median(y)
        elif self.loss_function is 'huber': return self._huber_initialise_predictions(y)
        else: return y.mean()   
    
    def _huber_initialise_predictions(self, y): #to start the preds
        best_loss = float('inf')
        ip = None
        for i in y:
            y_t = self._huber_loss(i) #loss at this point
            total_loss = self._huber_loss(y - y_t) #loss across all data using that point
            if total_loss < best_loss: #minimise loss over data
                best_loss = total_loss
                ip = i                 #get minimising initial prediction         
        return ip
    
    def _huber_gradient(self, residuals): #transforms residuals inot a huber gradient. 
        abs_resid = np.abs(residuals)
        gradient = np.where(abs_resid <= self.epsilon, -residuals, -self.epsilon * np.sign(residuals))
        return gradient
    
    def _mse_(self, residuals, left_mask, right_mask):
        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        mse = (np.mean((residuals[left_mask] - left_value) ** 2) * np.sum(left_mask) + 
           np.mean((residuals[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
        return mse, left_value, right_value

    def _abs_error(self, residuals, left_mask, right_mask): #note, residuals have to be median, not mean for abs_error: outliers
        sign_vector = np.sign(residuals)

        left_value = sign_vector[left_mask].mean()
        right_value = sign_vector[right_mask].mean()

        mae = (np.mean((sign_vector[left_mask] - left_value) ** 2) * np.sum(left_mask) +
               np.mean((sign_vector[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)

        return mae, left_value, right_value

    def _friedman_MSE(self, residuals, left_mask, right_mask, weights = None):
        if weights is None:
            weights = np.ones(len(residuals))

        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        w_l = np.sum(weights[left_mask]) 
        w_r = np.sum(weights[right_mask])

        mean_l = np.mean(residuals[left_mask])
        mean_r = np.mean(residuals[right_mask])

        friedman_mse = ((w_l * w_r) / (w_l + w_r)) * ((mean_l - mean_r)**2)

        return friedman_mse, left_value, right_value

    def _huber(self, residuals, left_mask, right_mask):
        huber_residuals = self._huber_gradient(residuals) #always huber-ise the residuals, even when already hubered from r-1 

        left_value = np.mean(huber_residuals[left_mask]) if np.any(left_mask) else 0
        right_value = np.mean(huber_residuals[right_mask]) if np.any(right_mask) else 0
        #need to figure out best huber loss over all predictions. 
        y_pred = np.full_like(huber_residuals, left_value)
        y_pred[right_mask] = right_value
        
        error = self._huber_loss(self._compute_residuals(huber_residuals, y_pred))
        
        return error, left_value, right_value    

    def _train_weak_learner(self, X, residuals):
        X = np.array(X)  # Ensure X is a NumPy array
        n_samples, n_features = X.shape
        best_feature_index = None
        best_threshold = None
        best_left_value = None
        best_right_value = None

        if self.loss_function is 'friedmanMSE': best_error = float('-inf')
        else: best_error = float('inf')

        # Iterate over each feature
        for feature_index in range(n_features):
            thresholds = np.unique(X[:, feature_index])

            # Test each threshold as a potential split point
            for threshold in thresholds:
                left_mask = X[:, feature_index] <= threshold
                right_mask = X[:, feature_index] > threshold

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                    
                #skip if either side has fewer samples than the specified minimum samples per leaf
                if (np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf): continue
                    
                #impurity calculation
                if self.loss_function is 'MSE': error, left_value, right_value =  self._mse_(residuals, left_mask, right_mask)
                elif self.loss_function is 'friedmanMSE': error, left_value, right_value =  self._friedman_MSE(residuals, left_mask, right_mask)
                elif self.loss_function is 'MAE': error, left_value, right_value =  self._abs_error(residuals, left_mask, right_mask)
                elif self.loss_function is 'huber': error, left_value, right_value = self._huber(residuals, left_mask, right_mask)

                    
    #             If this is the best split so far, store it. FreidmanMSE maximises, so we need a different approach
                if self.loss_function is 'friedmanMSE':
                    if error > best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value
                else:
                    if error < best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value

        return best_feature_index, best_threshold, best_left_value, best_right_value, best_error
    
    def _fit_stump(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            stump = self._train_weak_learner(X[indices], resid_subsample)
            feature_index, threshold, left_value, right_value, best_error = stump

            current_pred_subsample = self.current_pred[indices]
            updates = self._update_predictions(current_pred_subsample, feature_index, threshold, left_value, right_value, X[indices])
            self.current_pred[indices] = updates
            #========================
        else: 
            stump = self._train_weak_learner(X, residuals)
            feature_index, threshold, left_value, right_value, best_error = stump
            self.current_pred = self._update_predictions(self.current_pred, feature_index, threshold, left_value, right_value, X)
        #========================
        self.stumps.append(stump)
        self.loss_history.append(best_error)
        return self
    
    #=======================================TREE FUNCTIONS=======================================
    def _leaf_value(self, y): return np.mean(y) #y is residuals for GBoost, can use initialise because the prediction is the mean
    
    def _grow_tree(self, X, residuals, depth = 0): #recursively grow the decision tree up to the specified depth. Which is a stump if not specified
        n_samples, n_features = X.shape
        #n_classes = len(np.unique(y)) #use this to stop if there are too few values
        
        #check stopping criteria
        if (depth >= self.max_depth or n_samples < self.min_samples_split): # or n_classes == 1):
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        #find best split
        best_feature, best_threshold, left_value, right_value, best_error = self._train_weak_learner(X, residuals)
        
        #if no split found, create a leaf
        if best_feature is None: 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        self.current_pred = self._update_predictions(self.current_pred, best_feature, best_threshold, left_value, right_value, X_train)
        
        #split data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold
        left_subtree = self._grow_tree(X[left_mask], residuals[left_mask], depth +1)
        right_subtree = self._grow_tree(X[right_mask], residuals[right_mask], depth +1)
                
        #track the tree stats
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'left value': left_value,
            'right value': right_value,
            'left': left_subtree,
            'right': right_subtree,
            'error': best_error,
            'is_leaf': False
        }
    
    def _fit_tree(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            self.root = self._grow_tree(X[indices], resid_subsample)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.error.loc[t,'error'] = self.root['error']
            self.trees.append(tree)
            #========================
        else:
            self.root = self._grow_tree(X, residuals)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.loss_history.append(best_error)
            self.trees.append(tree)
        return self
            
    #=====================================+==MLP FUNCTIONS=======================================
    def _initialize_parameters(self, n_features):
        """Initialize weights and biases for each layer using He initialization."""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) +[1]  # Output layer has 1 neuron for regression
        self.weights = []
        self.biases = []
        
        #adam
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i+1]
            # He initialization for ReLU
            if self.activation == 'relu':
                limit = np.sqrt(2.0 / fan_in)
                W = np.random.normal(0, limit, (fan_in, fan_out))
            else:
                # Xavier/Glorot for other activations
                limit = np.sqrt(6.0 / (fan_in + fan_out))
                W = np.random.uniform(-limit, limit, (fan_in, fan_out))
            b = np.zeros((1, fan_out))
            self.weights.append(W)
            self.biases.append(b)
            
            #adam
            self.m_weights.append(np.zeros((fan_in, fan_out)))
            self.v_weights.append(np.zeros((fan_in, fan_out)))
            self.m_biases.append(np.zeros((1, fan_out)))
            self.v_biases.append(np.zeros((1, fan_out)))
    
    def _forward_pass(self, X):
        """Perform forward propagation through the network."""
        activations = [X]
        zs = []  # Linear combinations before activation
        
        # Forward pass through hidden layers
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            zs.append(z)
            # Apply activation function
            if self.activation == 'relu':
                a = np.maximum(0, z)
            elif self.activation == 'tanh':
                a = np.tanh(z)
            elif self.activation == 'sigmoid':
                a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            else:
                raise ValueError(f"Unsupported activation function: {self.activation}")
            activations.append(a)
        
        # Output layer (no activation for regression)
        z_output = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        zs.append(z_output)
        y_pred = z_output  # Linear output for regression
        activations.append(y_pred)
        
        return activations, zs
    
    def _compute_loss(self, y, residuals):
        """Compute mean squared error loss.""" #put loss in here, then check if loss is better or not
        if self.loss_function is 'MSE': return np.mean((y - residuals) ** 2)
        elif self.loss_function is 'MAE': return np.mean(abs(y - residuals))
        elif self.loss_function is 'huber': return _huber_loss(residuals)        
    
    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def _backward_pass(self, X, y, activations, zs):
        """Perform backpropagation to compute gradients."""
        m = X.shape[0]  # Number of samples
        deltas = []
        
        # Output layer delta (for regression, derivative of MSE w.r.t. z)
        delta_output = (activations[-1] - y) / m
        deltas.append(delta_output)
        
        # Backpropagate through hidden layers
        for l in range(len(self.weights) - 2, -1, -1):
            # Delta for current layer
            delta = np.dot(deltas[-1], self.weights[l+1].T)
            # Apply derivative of activation function
            if self.activation == 'relu':
                delta *= (zs[l] > 0).astype(float)
            elif self.activation == 'tanh':
                delta *= (1 - np.tanh(zs[l]) ** 2)
            elif self.activation == 'sigmoid':
                s = 1 / (1 + np.exp(-np.clip(zs[l], -500, 500)))
                delta *= s * (1 - s)
            deltas.append(delta)
        
        # Reverse deltas to match layer order
        deltas.reverse()
        
        # Compute gradients for weights and biases
        for l in range(len(self.weights)):
            # Gradient of loss w.r.t. weights
            dW = np.dot(activations[l].T, deltas[l]) / m
            # Gradient of loss w.r.t. biases
            db = np.sum(deltas[l], axis=0, keepdims=True) / m
            # Update weights and biases
            self.weights[l] -= self.alpha * dW # * self.learning_rate
            self.biases[l] -= self.alpha * db # * self.learning_rate
    
    def _get_shuffled(self, n_samples, X, y):
        permutation = np.random.permutation(n_samples)
        X_shuffled = X[permutation]
        y_shuffled = y[permutation]
        return X_shuffled, y_shuffled

    
    def _SGD(self, X_batch, y_batch, t, n_samples, n_features, preds):
         for epoch in range(self.n_epochs):
            # shuffle datasets
#             permutation = np.random.permutation(n_samples)
#             X_shuffled = X[permutation]
#             y_shuffled = y[permutation]
            X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)

            # mini-batch loop
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]

                activations, zs = self._forward_pass(X_batch)
                y_pred = activations[-1]

                delta = y_pred - y_batch
                dW = np.dot(activations[-2].T, delta) / X_batch.shape[0]
                db = np.sum(delta, axis=0) / X_batch.shape[0]
                self.weights[-1] -= self.learning_rate * dW
                self.biases[-1] -= self.learning_rate * db

                for l in range(len(self.weights) - 2, -1, -1):
                    delta = np.dot(delta, self.weights[l+1].T) * self._relu_derivative(zs[l]) # d_activation(z)
                    dW = np.dot(activations[l].T, delta) / X_batch.shape[0]
                    db = np.sum(delta, axis=0) / X_batch.shape[0]

                    self.weights[l] -= self.learning_rate * dW
                    self.biases[l] -= self.learning_rate * db

            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])

            activations, _ = self._forward_pass(X)
            epoch_loss = self._compute_loss(y, activations[-1])
            self.loss_history.append(epoch_loss)
            
            preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
            self.estimator_preds.append(preds) # store the preds

            return self
    
    def _adam(self, X, y, t, n_samples, n_features, preds):
           # global time step for Adam bias correction
        time = 0
        for epoch in range(self.n_epochs):
            # Shuffle data: Stochastic element to avoid getting stuck-----------------------------------
#             permutation = np.random.permutation(n_samples)
#             X_shuffled = X[permutation]
#             y_shuffled = y[permutation]
            X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)

            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]

                time += 1

                # 1. forward pass
                activations, zs = self._forward_pass(X_batch)
                y_pred = activations[-1] # Output of the network

                # 2. backpropagation (using Adam slightly different than SGD)
                delta = y_pred - y_batch
                dW = np.dot(activations[-2].T, delta) / X_batch.shape[0] # Average over batch
                db = np.sum(delta, axis=0) / X_batch.shape[0]

                # apply Adam updates to weights
                self.m_weights[-1] = self.beta2 * self.m_weights[-1] + (1 - self.beta2) * dW
                self.v_weights[-1] = self.beta3 * self.v_weights[-1] + (1 - self.beta3) * (dW ** 2)
                m_w_hat = self.m_weights[-1] / (1 - self.beta2**t)
                v_w_hat = self.v_weights[-1] / (1 - self.beta3**t)
                self.weights[-1] -= (self.alpha * m_w_hat ) / (np.sqrt(v_w_hat) + self.eta)

                # apply Adam updates to bias
                self.m_biases[-1] = self.beta2 * self.m_biases[-1] + (1 - self.beta2) * db
                self.v_biases[-1] = self.beta3 * self.v_biases[-1] + (1 - self.beta3) * (db ** 2)
                m_b_hat = self.m_biases[-1] / (1 - self.beta2**t)
                v_b_hat = self.v_biases[-1] / (1 - self.beta3**t)
                self.biases[-1] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

                # Propagate gradients backward through hidden layers
                for l in range(len(self.weights) - 2, -1, -1):
                    delta = np.dot(delta, self.weights[l+1].T) * self._relu_derivative(zs[l]) # d_activation(z)
                    dW_hidden = np.dot(activations[l].T, delta) / X_batch.shape[0]
                    db_hidden = np.sum(delta, axis=0) / X_batch.shape[0]

                    # apply Adam updates to weights
                    self.m_weights[l] = self.beta2 * self.m_weights[l] + (1 - self.beta2) * dW_hidden
                    self.v_weights[l] = self.beta3 * self.v_weights[l] + (1 - self.beta3) * (dW_hidden ** 2)
                    m_w_hat = self.m_weights[l] / (1 - self.beta2**t)
                    v_w_hat = self.v_weights[l] / (1 - self.beta3**t)
                    self.weights[l] -= (self.alpha * m_w_hat) / (np.sqrt(v_w_hat) + self.eta)

                    # apply Adam updates to bias
                    self.m_biases[l] = self.beta2 * self.m_biases[l] + (1 - self.beta2) * db_hidden
                    self.v_biases[l] = self.beta3 * self.v_biases[l] + (1 - self.beta3) * (db_hidden ** 2)
                    m_b_hat = self.m_biases[l] / (1 - self.beta2**t)
                    v_b_hat = self.v_biases[l] / (1 - self.beta3**t)
                    self.biases[l] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

                self.weights_history.append([w.copy() for w in self.weights])
                self.biases_history.append([b.copy() for b in self.biases])

                activations, _ = self._forward_pass(X)
                epoch_loss = self._compute_loss(y, activations[-1])
                self.loss_history.append(epoch_loss)

                preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
                self.estimator_preds.append(preds) # store the preds

#                 if (epoch + 1) % 100 == 0:
#                     print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss:.4f}")
        return self
    
    def _fit_MLP(self, t, X, residuals): #activations are multipled by learning rate and alpha
        preds = {}
        X = np.asarray(X)
        y = np.asarray(residuals).reshape(-1, 1) #changing to y for easy coding
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        #adam
        self.weights_history.append([w.copy() for w in self.weights])
        self.biases_history.append([b.copy() for b in self.biases])
        activations, _ = self._forward_pass(X)
        initial_loss = self._compute_loss(y, activations[-1])
        self.loss_history.append(initial_loss)
                
        # Training loop
        for epoch in range(self.n_epochs):
            # shuffle datasets
            permutation = np.random.permutation(n_samples)
            X_shuffled = X[permutation]
            y_shuffled = y[permutation]

            # mini-batch loop
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i : i + self.batch_size]
                y_batch = y_shuffled[i : i + self.batch_size]

                activations, zs = self._forward_pass(X_batch)
                y_pred = activations[-1]

                if self.optimiser is 'SGD': self._SGD(X, y, t, n_samples, n_features, preds)
#             for epoch in range(self.n_epochs):
#                 # Shuffle data: Stochastic Gradient descent-----------------------------------
#                 permutation = np.random.permutation(n_samples)
#                 X_shuffled = X[permutation]
#                 y_shuffled = y[permutation]

#                 # Mini-batch loop
#                 for i in range(0, n_samples, self.batch_size):
#                     X_batch = X_shuffled[i:i + self.batch_size]
#                     y_batch = y_shuffled[i:i + self.batch_size]

#                     # Forward pass
#                     activations, zs = self._forward_pass(X_batch) #output for model prediction

#                     # Compute loss
#                     loss = self._compute_loss(y_batch, activations[-1])

#                     # Backward pass
#                     self._backward_pass(X_batch, y_batch, activations, zs)
#                 #Stochastic Gradient descent end----------------------------------------------
#                 activations, _ = self._forward_pass(X)
#                 preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
#                 self.estimator_preds.append(preds) # store the preds

#                 # Record loss every 100 epochs
#                 if (epoch + 1) % 100 == 0:
#                     epoch_loss = self._compute_loss(y, activations[-1])
#                     self.loss_history.append(epoch_loss)
#     #               print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss:.6f}")
        
                if self.optimiser is 'adam': self._adam(X, y, t, n_samples, n_features, preds)
             
        return self
    #=======================================Fit FUNCTIONS=======================================

    def _update_predictions(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.learning_rate * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.learning_rate * right_value) for i in range(len(X[:,feature_index]))])  
    
    def fit(self, X, y):
        self.initial_pred = self._initialize_predictions(y)
        self.current_pred = np.full(len(y), self.initial_pred)
                
        if self.classifier is 'stump':
            for t in range(self.n_estimators): 
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================  
                self._fit_stump(t, X, residuals)
                
        elif self.classifier is 'tree':
            for t in range(self.n_estimators): #growing t trees... ba-dump, tiss
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================
                self._fit_tree(t, X, residuals)
            
        elif self.classifier is 'MLP':
            X = StandardScaler().fit_transform(X)
            
            for t in range(self.n_estimators):
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================
                self._fit_MLP(t, X, residuals)
                
        return self
    
    def _predict_sample(self, x, node): #goes to bottom of tree to get prediction, will need to do update_predictions
        if node['is_leaf']: return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else: return self._predict_sample(x, node['right'])
    
    def predict(self, X):
        y_pred = np.full(len(X), self.initial_pred)
        
        if self.classifier is 'stump':
            for stump in self.stumps:
                y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
            return y_pred
          
        #iterate through each tree, get predictsion, weight them by learning rate and add the tree predictions to the total prediction (weight by gamme later)
        elif self.classifier is 'tree': 
            for t in range(self.n_estimators):
                predictions = [] #np.zeros(len(X))
                current_tree = self.trees[t][f'tree {t}']
                
                for x in X:
                    predictions.append(self._predict_sample(x, current_tree))
                predictions = np.array(predictions) * self.learning_rate
                y_pred += predictions
        
        elif self.classifier is 'MLP':
            X = StandardScaler().fit_transform(X)
#             print(self.activation_preds)
            for t in range(self.n_estimators):
                X = np.asarray(X)
                activations, _ = self._forward_pass(X)
                predictions = activations[-1].flatten()
                predictions *= self.learning_rate #updated during forward and back prop
                y_pred += predictions #add to y_pred
          
        return y_pred   
        
    def score(self, X, y): #for grid search and CV etc
        y_pred = self.predict(X)
        if self.score is 'MSE': return np.mean((y - y_pred)**2)
        elif self.score is 'RMSE': return np.sqrt(np.mean((y - y_pred) ** 2))
        elif self.score is 'MAE': return np.mean(np.abs(y - y_pred))
        elif self.score is 'MAPE': return np.mean(np.abs((y - y_pred) / y))
        elif self.score is 'R2': return 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))
        return 

Before packing backprops into funct

In [ ]:
%autoreload
class GBoost(BaseEstimator, RegressorMixin):
    def __init__(self, classifier = 'stump', n_estimators = 10, learning_rate = 0.5, max_depth = 1, 
                 subsample = None, epsilon = 0.5, beta = 0.5, min_samples_leaf = 0, min_samples_split = 0, 
                 loss_function = 'MSE', hidden_layer_sizes=(10,), activation='relu', optimiser =None,
                 beta2 =0.9, beta3 = 0.999, eta = 1e-8, alpha=0.01, n_epochs=1000, batch_size=32,
                 seed = 21, score = 'MSE'):
        
        self.classifier = classifier
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.epsilon = epsilon #for huber loss
        self.beta = beta #for huber loss
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.subsample = subsample
        self.loss_function = loss_function
        self.seed = seed
        self.score = score
        self.stumps = []
        self.loss_history = []  #records loss for plotting
        self.initial_pred = None
        self.current_pred = None
        self.root = None
        self.trees = []
        
        #MLP params
        self.hidden_layer_sizes = hidden_layer_sizes
        self.optimiser = optimiser
        self.activation = activation
        self.alpha = alpha
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.weights = []
        self.biases = []
        self.estimator_preds = []
        #adam
        self.beta2 = beta2
        self.beta3 = beta3
        self.eta = eta
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        self.weights_history = []
        self.biases_history = []
        
        # Set random seed if provided
        if self.seed:
            np.random.seed(self.seed)
        
    def _compute_residuals(self, y, y_pred): 
        if self.loss_function is 'huber': return self._huber_gradient((y - y_pred))
        else: return y - y_pred
        
    def _get_subsample(self, y): #return indices of sub sampled data
        np.random.seed(self.seed)
        return np.array(np.random.choice(len(y), size=(round(len(y) * self.subsample)), replace=False))
    
    def _huber_loss(self, residuals): #to find leaf values with huber loss minimisation
        abs_resid = np.abs(residuals)
        quadratic = self.beta * residuals**2
        linear = self.epsilon * (abs_resid - 0.5 * self.epsilon)

        return np.where(abs_resid <= self.epsilon, quadratic, linear).sum()
    
    def _initialize_predictions(self, y): #also use for leaf value
        if self.loss_function is 'MAE': return np.median(y)
        elif self.loss_function is 'huber': return self._huber_initialise_predictions(y)
        else: return y.mean()   
    
    def _huber_initialise_predictions(self, y): #to start the preds
        best_loss = float('inf')
        ip = None
        for i in y:
            y_t = self._huber_loss(i) #loss at this point
            total_loss = self._huber_loss(y - y_t) #loss across all data using that point
            if total_loss < best_loss: #minimise loss over data
                best_loss = total_loss
                ip = i                 #get minimising initial prediction         
        return ip
    
    def _huber_gradient(self, residuals): #transforms residuals inot a huber gradient. 
        abs_resid = np.abs(residuals)
        gradient = np.where(abs_resid <= self.epsilon, -residuals, -self.epsilon * np.sign(residuals))
        return gradient
    
    def _mse_(self, residuals, left_mask, right_mask):
        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        mse = (np.mean((residuals[left_mask] - left_value) ** 2) * np.sum(left_mask) + 
           np.mean((residuals[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
        return mse, left_value, right_value

    def _abs_error(self, residuals, left_mask, right_mask): #note, residuals have to be median, not mean for abs_error: outliers
        sign_vector = np.sign(residuals)

        left_value = sign_vector[left_mask].mean()
        right_value = sign_vector[right_mask].mean()

        mae = (np.mean((sign_vector[left_mask] - left_value) ** 2) * np.sum(left_mask) +
               np.mean((sign_vector[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)

        return mae, left_value, right_value

    def _friedman_MSE(self, residuals, left_mask, right_mask, weights = None):
        if weights is None:
            weights = np.ones(len(residuals))

        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        w_l = np.sum(weights[left_mask]) 
        w_r = np.sum(weights[right_mask])

        mean_l = np.mean(residuals[left_mask])
        mean_r = np.mean(residuals[right_mask])

        friedman_mse = ((w_l * w_r) / (w_l + w_r)) * ((mean_l - mean_r)**2)

        return friedman_mse, left_value, right_value

    def _huber(self, residuals, left_mask, right_mask):
        huber_residuals = self._huber_gradient(residuals) #always huber-ise the residuals, even when already hubered from r-1 

        left_value = np.mean(huber_residuals[left_mask]) if np.any(left_mask) else 0
        right_value = np.mean(huber_residuals[right_mask]) if np.any(right_mask) else 0
        #need to figure out best huber loss over all predictions. 
        y_pred = np.full_like(huber_residuals, left_value)
        y_pred[right_mask] = right_value
        
        error = self._huber_loss(self._compute_residuals(huber_residuals, y_pred))
        
        return error, left_value, right_value    

    def _train_weak_learner(self, X, residuals):
        X = np.array(X)  # Ensure X is a NumPy array
        n_samples, n_features = X.shape
        best_feature_index = None
        best_threshold = None
        best_left_value = None
        best_right_value = None

        if self.loss_function is 'friedmanMSE': best_error = float('-inf')
        else: best_error = float('inf')

        # Iterate over each feature
        for feature_index in range(n_features):
            thresholds = np.unique(X[:, feature_index])

            # Test each threshold as a potential split point
            for threshold in thresholds:
                left_mask = X[:, feature_index] <= threshold
                right_mask = X[:, feature_index] > threshold

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                    
                #skip if either side has fewer samples than the specified minimum samples per leaf
                if (np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf): continue
                    
                #impurity calculation
                if self.loss_function is 'MSE': error, left_value, right_value =  self._mse_(residuals, left_mask, right_mask)
                elif self.loss_function is 'friedmanMSE': error, left_value, right_value =  self._friedman_MSE(residuals, left_mask, right_mask)
                elif self.loss_function is 'MAE': error, left_value, right_value =  self._abs_error(residuals, left_mask, right_mask)
                elif self.loss_function is 'huber': error, left_value, right_value = self._huber(residuals, left_mask, right_mask)

                    
    #             If this is the best split so far, store it. FreidmanMSE maximises, so we need a different approach
                if self.loss_function is 'friedmanMSE':
                    if error > best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value
                else:
                    if error < best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value

        return best_feature_index, best_threshold, best_left_value, best_right_value, best_error
    
    def _fit_stump(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            stump = self._train_weak_learner(X[indices], resid_subsample)
            feature_index, threshold, left_value, right_value, best_error = stump

            current_pred_subsample = self.current_pred[indices]
            updates = self._update_predictions(current_pred_subsample, feature_index, threshold, left_value, right_value, X[indices])
            self.current_pred[indices] = updates
            #========================
        else: 
            stump = self._train_weak_learner(X, residuals)
            feature_index, threshold, left_value, right_value, best_error = stump
            self.current_pred = self._update_predictions(self.current_pred, feature_index, threshold, left_value, right_value, X)
        #========================
        self.stumps.append(stump)
        self.loss_history.append(best_error)
        return self
    
    #=======================================TREE FUNCTIONS=======================================
    def _leaf_value(self, y): return np.mean(y) #y is residuals for GBoost, can use initialise because the prediction is the mean
    
    def _grow_tree(self, X, residuals, depth = 0): #recursively grow the decision tree up to the specified depth. Which is a stump if not specified
        n_samples, n_features = X.shape
        #n_classes = len(np.unique(y)) #use this to stop if there are too few values
        
        #check stopping criteria
        if (depth >= self.max_depth or n_samples < self.min_samples_split): # or n_classes == 1):
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        #find best split
        best_feature, best_threshold, left_value, right_value, best_error = self._train_weak_learner(X, residuals)
        
        #if no split found, create a leaf
        if best_feature is None: 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        self.current_pred = self._update_predictions(self.current_pred, best_feature, best_threshold, left_value, right_value, X_train)
        
        #split data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold
        left_subtree = self._grow_tree(X[left_mask], residuals[left_mask], depth +1)
        right_subtree = self._grow_tree(X[right_mask], residuals[right_mask], depth +1)
                
        #track the tree stats
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'left value': left_value,
            'right value': right_value,
            'left': left_subtree,
            'right': right_subtree,
            'error': best_error,
            'is_leaf': False
        }
    
    def _fit_tree(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            self.root = self._grow_tree(X[indices], resid_subsample)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.error.loc[t,'error'] = self.root['error']
            self.trees.append(tree)
            #========================
        else:
            self.root = self._grow_tree(X, residuals)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.loss_history.append(best_error)
            self.trees.append(tree)
        return self
            
    #=====================================+==MLP FUNCTIONS=======================================
    def _initialize_parameters(self, n_features):
        """Initialize weights and biases for each layer using He initialization."""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) +[1]  # Output layer has 1 neuron for regression
        self.weights = []
        self.biases = []
        
        #adam
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i+1]
            # He initialization for ReLU
            if self.activation == 'relu':
                limit = np.sqrt(2.0 / fan_in)
                W = np.random.normal(0, limit, (fan_in, fan_out))
            else:
                # Xavier/Glorot for other activations
                limit = np.sqrt(6.0 / (fan_in + fan_out))
                W = np.random.uniform(-limit, limit, (fan_in, fan_out))
            b = np.zeros((1, fan_out))
            self.weights.append(W)
            self.biases.append(b)
            
            #adam
            self.m_weights.append(np.zeros((fan_in, fan_out)))
            self.v_weights.append(np.zeros((fan_in, fan_out)))
            self.m_biases.append(np.zeros((1, fan_out)))
            self.v_biases.append(np.zeros((1, fan_out)))
    
    def _forward_pass(self, X):
        """Perform forward propagation through the network."""
        activations = [X]
        zs = []  # Linear combinations before activation
        
        # Forward pass through hidden layers
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            zs.append(z)
            # Apply activation function
            if self.activation == 'relu':
                a = np.maximum(0, z)
            elif self.activation == 'tanh':
                a = np.tanh(z)
            elif self.activation == 'sigmoid':
                a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            else:
                raise ValueError(f"Unsupported activation function: {self.activation}")
            activations.append(a)
        
        # Output layer (no activation for regression)
        z_output = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        zs.append(z_output)
        y_pred = z_output  # Linear output for regression
        activations.append(y_pred)
        
        return activations, zs
    
    def _compute_loss(self, y, residuals):
        """Compute mean squared error loss.""" #put loss in here, then check if loss is better or not
        if self.loss_function is 'MSE': return np.mean((y - residuals) ** 2)
        elif self.loss_function is 'MAE': return np.mean(abs(y - residuals))
        elif self.loss_function is 'huber': return _huber_loss(residuals)        
    
    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def _backward_pass(self, X, y, activations, zs):
        """Perform backpropagation to compute gradients."""
        m = X.shape[0]  # Number of samples
        deltas = []
        
        if self.optimiser is None:
            # Output layer delta (for regression, derivative of MSE w.r.t. z)
            delta_output = (activations[-1] - y) / m
            deltas.append(delta_output)

            # Backpropagate through hidden layers
            for l in range(len(self.weights) - 2, -1, -1):
                # Delta for current layer
                delta = np.dot(deltas[-1], self.weights[l+1].T)
                # Apply derivative of activation function
                if self.activation == 'relu':
                    delta *= (zs[l] > 0).astype(float)
                elif self.activation == 'tanh':
                    delta *= (1 - np.tanh(zs[l]) ** 2)
                elif self.activation == 'sigmoid':
                    s = 1 / (1 + np.exp(-np.clip(zs[l], -500, 500)))
                    delta *= s * (1 - s)
                deltas.append(delta)

            # Reverse deltas to match layer order
            deltas.reverse()

            # Compute gradients for weights and biases
            for l in range(len(self.weights)):
                # Gradient of loss w.r.t. weights
                dW = np.dot(activations[l].T, deltas[l]) / m
                # Gradient of loss w.r.t. biases
                db = np.sum(deltas[l], axis=0, keepdims=True) / m
                # Update weights and biases
                self.weights[l] -= self.alpha * dW # * self.learning_rate
                self.biases[l] -= self.alpha * db # * self.learning_rate
                
        elif self.optimiser is 'SGD':
            y_pred = activations[-1]

            delta = y_pred - y
            dW = np.dot(activations[-2].T, delta) / X.shape[0]
            db = np.sum(delta, axis=0) / X.shape[0]
            self.weights[-1] -= self.learning_rate * dW
            self.biases[-1] -= self.learning_rate * db

            for l in range(len(self.weights) - 2, -1, -1):
                delta = np.dot(delta, self.weights[l+1].T) * self._relu_derivative(zs[l]) # d_activation(z)
                dW = np.dot(activations[l].T, delta) / X.shape[0]
                db = np.sum(delta, axis=0) / X.shape[0]

                self.weights[l] -= self.learning_rate * dW
                self.biases[l] -= self.learning_rate * db
                
            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])
#             self._store_preds(X, y, t, epoch) #store preds
                
        return self
        
            
    def _get_shuffled(self, n_samples, X, y):
        permutation = np.random.permutation(n_samples)
        X_shuffled = X[permutation]
        y_shuffled = y[permutation]
        return X_shuffled, y_shuffled
    
    def _set_epoch_loss(self, y, activation):
        epoch_loss = self._compute_loss(y, activation)
        self.loss_history.append(epoch_loss)
        return self
    
    def _SGD():
        return
    
    def _adam():
        return
    
    def _store_preds(self, X, y, t, epoch):
        preds = {}
        activations, _ = self._forward_pass(X)
        self._set_epoch_loss(y, activations[-1])
        preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
        self.estimator_preds.append(preds) # store the preds
        return self
    
    def _fit_MLP(self, t, X, residuals): #activations are multipled by learning rate and alpha
        preds = {} #to store each epoch and iteration's predictions
        X = np.asarray(X)
        y = np.asarray(residuals).reshape(-1, 1) #changing to y for easy coding
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        self.weights_history.append([w.copy() for w in self.weights])
        self.biases_history.append([b.copy() for b in self.biases])
        activations, _ = self._forward_pass(X)
        initial_loss = self._compute_loss(y, activations[-1])
        self.loss_history.append(initial_loss)
        
        # global time step for Adam bias correction
        time = 0
        
        # Training loop
        if self.optimiser is None:
            for epoch in range(self.n_epochs):
                # Shuffle data: -------------------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)

                # Mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i:i + self.batch_size]
                    y_batch = y_shuffled[i:i + self.batch_size]

                    # Forward pass
                    activations, zs = self._forward_pass(X_batch) #output for model prediction

                    # Backward pass
                    self._backward_pass(X_batch, y_batch, activations, zs)
                #end loop---------------------------------------------------
                self._store_preds(X, y, t, epoch)
        
        if self.optimiser is 'SGD':
            for epoch in range(self.n_epochs):
            # shuffle datasets
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)

                # mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]

                    activations, zs = self._forward_pass(X_batch)
                    self._backward_pass(X_batch, y_batch, activations, zs)
#                     y_pred = activations[-1]

#                     delta = y_pred - y_batch
#                     dW = np.dot(activations[-2].T, delta) / X_batch.shape[0]
#                     db = np.sum(delta, axis=0) / X_batch.shape[0]
#                     self.weights[-1] -= self.learning_rate * dW
#                     self.biases[-1] -= self.learning_rate * db

#                     for l in range(len(self.weights) - 2, -1, -1):
#                         delta = np.dot(delta, self.weights[l+1].T) * self._relu_derivative(zs[l]) # d_activation(z)
#                         dW = np.dot(activations[l].T, delta) / X_batch.shape[0]
#                         db = np.sum(delta, axis=0) / X_batch.shape[0]

#                         self.weights[l] -= self.learning_rate * dW
#                         self.biases[l] -= self.learning_rate * db

#                 self.weights_history.append([w.copy() for w in self.weights])
#                 self.biases_history.append([b.copy() for b in self.biases])
                #store results
                self._store_preds(X, y, t, epoch)

        if self.optimiser is 'adam':
             for epoch in range(self.n_epochs):
                # Shuffle data:  Gradient descent-----------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)
                
                for i in range(0, n_samples, self.batch_size):
                    time += 1
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]

                    # 1. forward pass
                    activations, zs = self._forward_pass(X_batch)
                    y_pred = activations[-1] # Output of the network

                    # 2. backpropagation (using Adam slightly different)
                    delta = y_pred - y_batch
                    dW = np.dot(activations[-2].T, delta) / X_batch.shape[0] # Average over batch
                    db = np.sum(delta, axis=0) / X_batch.shape[0]

                    # apply Adam updates to weights
                    self.m_weights[-1] = self.beta2 * self.m_weights[-1] + (1 - self.beta2) * dW
                    self.v_weights[-1] = self.beta3 * self.v_weights[-1] + (1 - self.beta3) * (dW ** 2)
                    m_w_hat = self.m_weights[-1] / (1 - self.beta2**time)
                    v_w_hat = self.v_weights[-1] / (1 - self.beta3**time)
                    self.weights[-1] -= (self.alpha * m_w_hat ) / (np.sqrt(v_w_hat) + self.eta)

                    # apply Adam updates to bias
                    self.m_biases[-1] = self.beta2 * self.m_biases[-1] + (1 - self.beta2) * db
                    self.v_biases[-1] = self.beta3 * self.v_biases[-1] + (1 - self.beta3) * (db ** 2)
                    m_b_hat = self.m_biases[-1] / (1 - self.beta2**time)
                    v_b_hat = self.v_biases[-1] / (1 - self.beta3**time)
                    self.biases[-1] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

                    # Propagate gradients backward through hidden layers
                    for l in range(len(self.weights) - 2, -1, -1):
                        delta = np.dot(delta, self.weights[l+1].T) * self._relu_derivative(zs[l]) # d_activation(z)
                        dW_hidden = np.dot(activations[l].T, delta) / X_batch.shape[0]
                        db_hidden = np.sum(delta, axis=0) / X_batch.shape[0]

                        # apply Adam updates to weights
                        self.m_weights[l] = self.beta2 * self.m_weights[l] + (1 - self.beta2) * dW_hidden
                        self.v_weights[l] = self.beta3 * self.v_weights[l] + (1 - self.beta3) * (dW_hidden ** 2)
                        m_w_hat = self.m_weights[l] / (1 - self.beta2**time)
                        v_w_hat = self.v_weights[l] / (1 - self.beta3**time)
                        self.weights[l] -= (self.alpha * m_w_hat) / (np.sqrt(v_w_hat) + self.eta)

                        # apply Adam updates to bias
                        self.m_biases[l] = self.beta2 * self.m_biases[l] + (1 - self.beta2) * db_hidden
                        self.v_biases[l] = self.beta3 * self.v_biases[l] + (1 - self.beta3) * (db_hidden ** 2)
                        m_b_hat = self.m_biases[l] / (1 - self.beta2**time)
                        v_b_hat = self.v_biases[l] / (1 - self.beta3**time)
                        self.biases[l] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

                    self.weights_history.append([w.copy() for w in self.weights])
                    self.biases_history.append([b.copy() for b in self.biases])
                    #store results
                    self._store_preds(X, y, t, epoch)

        return self
    #=======================================Fit FUNCTIONS=======================================

    def _update_predictions(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.learning_rate * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.learning_rate * right_value) for i in range(len(X[:,feature_index]))])  
    
    def fit(self, X, y):
        self.initial_pred = self._initialize_predictions(y)
        self.current_pred = np.full(len(y), self.initial_pred)
                
        if self.classifier is 'stump':
            for t in range(self.n_estimators): 
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================  
                self._fit_stump(t, X, residuals)
                
        elif self.classifier is 'tree':
            for t in range(self.n_estimators): #growing t trees... ba-dump, tiss
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================
                self._fit_tree(t, X, residuals)
            
        elif self.classifier is 'MLP':
            X_test_scaled = StandardScaler().fit_transform(X_test)
            
            for t in range(self.n_estimators):
                #========================  
                residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
                #========================
                self._fit_MLP(t, X, residuals)
                
        return self
    
    def _predict_sample(self, x, node): #goes to bottom of tree to get prediction, will need to do update_predictions
        if node['is_leaf']: return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else: return self._predict_sample(x, node['right'])
    
    def predict(self, X):
        y_pred = np.full(len(X), self.initial_pred)
        
        if self.classifier is 'stump':
            for stump in self.stumps:
                y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
            return y_pred
          
        #iterate through each tree, get predictsion, weight them by learning rate and add the tree predictions to the total prediction (weight by gamme later)
        elif self.classifier is 'tree': 
            for t in range(self.n_estimators):
                predictions = [] #np.zeros(len(X))
                current_tree = self.trees[t][f'tree {t}']
                
                for x in X:
                    predictions.append(self._predict_sample(x, current_tree))
                predictions = np.array(predictions) * self.learning_rate
                y_pred += predictions
        
        elif self.classifier is 'MLP':
            X = StandardScaler().fit_transform(X)
#             print(self.activation_preds)
            for t in range(self.n_estimators):
                X = np.asarray(X)
                activations, _ = self._forward_pass(X)
                predictions = activations[-1].flatten()
                predictions *= self.learning_rate #updated during forward and back prop
                y_pred += predictions #add to y_pred
          
        return y_pred   
        
    def score(self, X, y): #for grid search and CV etc
        y_pred = self.predict(X)
        if self.score is 'MSE': return np.mean((y - y_pred)**2)
        elif self.score is 'RMSE': return np.sqrt(np.mean((y - y_pred) ** 2))
        elif self.score is 'MAE': return np.mean(np.abs(y - y_pred))
        elif self.score is 'MAPE': return np.mean(np.abs((y - y_pred) / y))
        elif self.score is 'R2': return 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))
        return 

In [ ]:
%autoreload
class GBoost(BaseEstimator, RegressorMixin):
    def __init__(self, classifier = 'stump', n_estimators = 10, learning_rate = 0.5, max_depth = 1, 
                 subsample = None, epsilon = 0.5, beta = 0.5, min_samples_leaf = 0, min_samples_split = 0, 
                 loss_function = 'MSE', validation_fraction = None, patience = None, tolerance = None,
                 hidden_layer_sizes=(10,), activation='relu', optimiser = None,
                 beta2 =0.9, beta3 = 0.999, eta = 1e-8, alpha=0.01, n_epochs=1000, batch_size=None,
                 seed = 21, measure = 'MSE'):
        
        self.classifier = classifier
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.epsilon = epsilon #for huber loss
        self.beta = beta #for huber loss
        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.subsample = subsample
        self.loss_function = loss_function
        self.seed = seed
        self.measure = measure
        self.stumps = []
        self.loss_history = []  #records loss for plotting
        self.initial_pred = None
        self.current_pred = None
        self.root = None
        self.trees = []
        self.validation_fraction = validation_fraction
        self.validation_scores = []
        self.best_val_score = float('inf')
        self.patience = patience #(no iteration change)
        self.tolerance = tolerance
        self.best_iteration = 0
        self.no_improvement_count = 0
        
        #MLP params
        self.hidden_layer_sizes = hidden_layer_sizes
        self.optimiser = optimiser
        self.activation = activation
        self.alpha = alpha
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.weights = []
        self.biases = []
        self.estimator_preds = []
        self.delta_history = []
        self.weights_history = []
        self.biases_history = []
        #MLP adam
        self.beta2 = beta2
        self.beta3 = beta3
        self.eta = eta
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = [] 
        
        # Set random seed if provided
        if self.seed:
            np.random.seed(self.seed)
        
    def _compute_residuals(self, y, y_pred): 
        print(f'_compute_residuals: len y: {len(y)}, len y_pred {len(y_pred)}')
        if self.loss_function is 'huber': return self._huber_gradient((y - y_pred))
        else: return y - y_pred
        
    def _get_subsample(self, y): #return indices of sub sampled data
        np.random.seed(self.seed)
        return np.array(np.random.choice(len(y), size=(round(len(y) * self.subsample)), replace=False))
    
    def _huber_loss(self, residuals): #to find leaf values with huber loss minimisation
        abs_resid = np.abs(residuals)
        quadratic = self.beta * residuals**2
        linear = self.epsilon * (abs_resid - 0.5 * self.epsilon)

        return np.where(abs_resid <= self.epsilon, quadratic, linear).sum()
    
    def _initialize_predictions(self, y): #also use for leaf value
        if self.loss_function is 'MAE': return np.median(y)
        elif self.loss_function is 'huber': return self._huber_initialise_predictions(y)
        else: return y.mean()   
    
    def _huber_initialise_predictions(self, y): #to start the preds
        best_loss = float('inf')
        ip = None
        for i in y:
            y_t = self._huber_loss(i) #loss at this point
            total_loss = self._huber_loss(y - y_t) #loss across all data using that point
            if total_loss < best_loss: #minimise loss over data
                best_loss = total_loss
                ip = i                 #get minimising initial prediction         
        return ip
    
    def _huber_gradient(self, residuals): #transforms residuals inot a huber gradient. 
        abs_resid = np.abs(residuals)
        gradient = np.where(abs_resid <= self.epsilon, -residuals, -self.epsilon * np.sign(residuals))
        return gradient
    
    def _mse_(self, residuals, left_mask, right_mask):
        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        mse = (np.mean((residuals[left_mask] - left_value) ** 2) * np.sum(left_mask) + 
           np.mean((residuals[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
        return mse, left_value, right_value

    def _abs_error(self, residuals, left_mask, right_mask): #note, residuals have to be median, not mean for abs_error: outliers
        sign_vector = np.sign(residuals)

        left_value = sign_vector[left_mask].mean()
        right_value = sign_vector[right_mask].mean()

        mae = (np.mean((sign_vector[left_mask] - left_value) ** 2) * np.sum(left_mask) +
               np.mean((sign_vector[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)

        return mae, left_value, right_value

    def _friedman_MSE(self, residuals, left_mask, right_mask, weights = None):
        if weights is None:
            weights = np.ones(len(residuals))

        left_value = residuals[left_mask].mean()
        right_value = residuals[right_mask].mean()

        w_l = np.sum(weights[left_mask]) 
        w_r = np.sum(weights[right_mask])

        mean_l = np.mean(residuals[left_mask])
        mean_r = np.mean(residuals[right_mask])

        friedman_mse = ((w_l * w_r) / (w_l + w_r)) * ((mean_l - mean_r)**2)

        return friedman_mse, left_value, right_value

    def _huber(self, residuals, left_mask, right_mask):
        huber_residuals = self._huber_gradient(residuals) #always huber-ise the residuals, even when already hubered from r-1 

        left_value = np.mean(huber_residuals[left_mask]) if np.any(left_mask) else 0
        right_value = np.mean(huber_residuals[right_mask]) if np.any(right_mask) else 0
        #need to figure out best huber loss over all predictions. 
        y_pred = np.full_like(huber_residuals, left_value)
        y_pred[right_mask] = right_value
        
        error = self._huber_loss(self._compute_residuals(huber_residuals, y_pred))
        
        return error, left_value, right_value    

    def _train_weak_learner(self, X, residuals):
        X = np.array(X)  # Ensure X is a NumPy array
        n_samples, n_features = X.shape
        best_feature_index = None
        best_threshold = None
        best_left_value = None
        best_right_value = None

        if self.loss_function is 'friedmanMSE': best_error = float('-inf')
        else: best_error = float('inf')

        # Iterate over each feature
        for feature_index in range(n_features):
            thresholds = np.unique(X[:, feature_index])

            # Test each threshold as a potential split point
            for threshold in thresholds:
                left_mask = X[:, feature_index] <= threshold
                right_mask = X[:, feature_index] > threshold

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue
                    
                #skip if either side has fewer samples than the specified minimum samples per leaf
                if (np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf): continue
                    
                #impurity calculation
                if self.loss_function is 'MSE': error, left_value, right_value =  self._mse_(residuals, left_mask, right_mask)
                elif self.loss_function is 'friedmanMSE': error, left_value, right_value =  self._friedman_MSE(residuals, left_mask, right_mask)
                elif self.loss_function is 'MAE': error, left_value, right_value =  self._abs_error(residuals, left_mask, right_mask)
                elif self.loss_function is 'huber': error, left_value, right_value = self._huber(residuals, left_mask, right_mask)

                    
    #             If this is the best split so far, store it. FreidmanMSE maximises, so we need a different approach
                if self.loss_function is 'friedmanMSE':
                    if error > best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value
                else:
                    if error < best_error:
                        best_error = error
                        best_feature_index = feature_index
                        best_threshold = threshold
                        best_left_value = left_value
                        best_right_value = right_value

        return best_feature_index, best_threshold, best_left_value, best_right_value, best_error
    
    def _fit_stump(self, t, X, residuals):
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            resid_subsample = residuals[indices]

            stump = self._train_weak_learner(X[indices], resid_subsample)
            feature_index, threshold, left_value, right_value, best_error = stump

            current_pred_subsample = self.current_pred[indices]
            updates = self._update_predictions(current_pred_subsample, feature_index, threshold, left_value, right_value, X[indices])
            self.current_pred[indices] = updates
            #========================
        else: 
            stump = self._train_weak_learner(X, residuals)
            feature_index, threshold, left_value, right_value, best_error = stump
            self.current_pred = self._update_predictions(self.current_pred, feature_index, threshold, left_value, right_value, X)
        #========================
        self.stumps.append(stump)
        self.loss_history.append(best_error)
        return self
    
    #=======================================TREE FUNCTIONS=======================================
    def _leaf_value(self, y): return np.mean(y) #y is residuals for GBoost, can use initialise because the prediction is the mean
    
    def _grow_tree(self, X, residuals, indices, depth = 0): #recursively grow the decision tree up to the specified depth. Which is a stump if not specified
        n_samples, n_features = X.shape
#         print('_grow_tree X.shape: ', X.shape)
        
        #check stopping criteria
        if (depth >= self.max_depth or n_samples < self.min_samples_split): # or n_classes == 1):
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        #find best split
        best_feature, best_threshold, left_value, right_value, best_error = self._train_weak_learner(X, residuals)
        
        #if no split found, create a leaf
        if best_feature is None: 
            return {'value': self._leaf_value(residuals), 'is_leaf': True}
        
        print('_gorw_tree self.current_preds len: ', len(self.current_pred))
        print('_grow_tree indices: ', indices)
#         print('_gorw_tree self.current_preds[indices] len: ', len(self.current_pred[indices]))
        y_pred =  self.current_pred[indices]
#         print('_grow_tree y_pred:\n', y_pred)
        y_pred = self._update_predictions(y_pred, best_feature, best_threshold, left_value, right_value, X)
#         print('_grow_tree y_pred after update\n:', y_pred)
        self.current_pred[indices] = y_pred
#         print('_grow_tree current_pred after update\n:',  self.current_pred)
#         self.current_pred = self._update_predictions(self.current_pred, best_feature, best_threshold, left_value, right_value, X)
        
        #split data
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold
        left_subtree = self._grow_tree(X[left_mask], residuals[left_mask], depth +1)
        right_subtree = self._grow_tree(X[right_mask], residuals[right_mask], depth +1)
                
        #track the tree stats
        return {
            'feature': best_feature,
            'threshold': best_threshold,
            'left value': left_value,
            'right value': right_value,
            'left': left_subtree,
            'right': right_subtree,
            'error': best_error,
            'is_leaf': False
        }
    
    def _fit_tree(self, t, X, residuals):
        print('_fit_tree round ', t)
        if self.subsample is not None:
            indices = self._get_subsample(residuals) #get indices
            print('_fit_tree len indicies: ', len(indices))
            resid_subsample = residuals[indices]
            print('_fit_tree residuals len', len(residuals))
            print('_fit_tree indices len', len(indices))
            print('_fit_tree resid_subsample len', len(resid_subsample))
            self.root = self._grow_tree(X[indices], resid_subsample, indices)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.loss_history.append(self.root['error'])
            self.trees.append(tree)
            #========================
        else:
            self.root = self._grow_tree(X, residuals)
            tree = {}
            tree[f'tree {t}'] = self.root
            self.loss_history.append(self.root['error'])
            self.trees.append(tree)
        return self
            
    #=====================================+==MLP FUNCTIONS=======================================
    def _initialize_parameters(self, n_features):
        """Initialize weights and biases for each layer using He initialization."""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) +[1]  # Output layer has 1 neuron for regression
        self.weights = []
        self.biases = []
        
        #adam
        self.m_weights = []
        self.v_weights = []
        self.m_biases = []
        self.v_biases = []
        
        for i in range(len(layer_sizes) - 1):
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i+1]
            # He initialization for ReLU
            if self.activation == 'relu':
                limit = np.sqrt(2.0 / fan_in)
                W = np.random.normal(0, limit, (fan_in, fan_out))
            else:
                # Xavier/Glorot for other activations
                limit = np.sqrt(6.0 / (fan_in + fan_out))
                W = np.random.uniform(-limit, limit, (fan_in, fan_out))
            b = np.zeros((1, fan_out))
            self.weights.append(W)
            self.biases.append(b)
            
            #adam
            self.m_weights.append(np.zeros((fan_in, fan_out)))
            self.v_weights.append(np.zeros((fan_in, fan_out)))
            self.m_biases.append(np.zeros((1, fan_out)))
            self.v_biases.append(np.zeros((1, fan_out)))
    
    def _forward_pass(self, X):
        """Perform forward propagation through the network."""
        activations = [X]
        zs = []  # Linear combinations before activation
        
        # Forward pass through hidden layers
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            zs.append(z)
            # Apply activation function
            if self.activation == 'relu':
                a = np.maximum(0, z)
            elif self.activation == 'tanh':
                a = np.tanh(z)
            elif self.activation == 'sigmoid':
                a = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            else:
                raise ValueError(f"Unsupported activation function: {self.activation}")
            activations.append(a)
        
        # Output layer (no activation for regression)
        z_output = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        zs.append(z_output)
        y_pred = z_output  # Linear output for regression
        activations.append(y_pred)
        
        return activations, zs
    
    def _compute_loss(self, y, residuals):
        """Compute mean squared error loss.""" #put loss in here, then check if loss is better or not
        if self.loss_function is 'MSE': return np.mean((y - residuals) ** 2)
        elif self.loss_function is 'MAE': return np.mean(abs(y - residuals))
        elif self.loss_function is 'huber': return _huber_loss(residuals)        
    
    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def _get_delta_update(self, delta, weights, z):
        if self.activation == 'relu': return np.dot(delta, weights.T) * self._relu_derivative(z)
        elif self.activation == 'tanh': return np.dot(delta, weights.T) * (z > 0).astype(float)
        elif self.activation == 'sigmoid':
            s = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
            return np.dot(delta, weights.T) * s * (1-s)
            
    
    def _backward_pass(self, X, y, activations, zs, time=None):
        """Perform backpropagation to compute gradients."""
        m = X.shape[0]  # Number of samples
        deltas = []
        
        if self.optimiser is None:
            # Output layer delta (for regression, derivative of MSE w.r.t. z)
            delta = (activations[-1] - y) / m
            deltas.append(delta)

            # Backpropagate through hidden layers
            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)

            # Reverse deltas to match layer order
            deltas.reverse()
            self.delta_history.append(deltas)
            
            # Compute gradients for weights and biases
            for l in range(len(self.weights)):
                # Gradient of loss w.r.t. weights
                dW = np.dot(activations[l].T, deltas[l]) / m
                # Gradient of loss w.r.t. biases
                db = np.sum(deltas[l], axis=0, keepdims=True) / m
                # Update weights and biases
                self.weights[l] -= self.alpha * dW # * self.learning_rate
                self.biases[l] -= self.alpha * db # * self.learning_rate
            
        elif self.optimiser is 'SGD':
            y_pred = activations[-1]

            delta = y_pred - y
            dW = np.dot(activations[-2].T, delta) / X.shape[0]
            db = np.sum(delta, axis=0) / X.shape[0]
            self.weights[-1] -= self.learning_rate * dW
            self.biases[-1] -= self.learning_rate * db

            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)
                self.delta_history.append(deltas)
                dW = np.dot(activations[l].T, delta) / X.shape[0]
                db = np.sum(delta, axis=0) / X.shape[0]

                self.weights[l] -= self.learning_rate * dW
                self.biases[l] -= self.learning_rate * db
                
            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])

        elif self.optimiser is 'adam':
            y_pred = activations[-1] # Output of the network

            # 2. backpropagation (using Adam slightly different)
            delta = y_pred - y
            dW = np.dot(activations[-2].T, delta) / X.shape[0] # Average over batch
            db = np.sum(delta, axis=0) / X.shape[0]

            # apply Adam updates to weights
            self.m_weights[-1] = self.beta2 * self.m_weights[-1] + (1 - self.beta2) * dW
            self.v_weights[-1] = self.beta3 * self.v_weights[-1] + (1 - self.beta3) * (dW ** 2)
            m_w_hat = self.m_weights[-1] / (1 - self.beta2**time)
            v_w_hat = self.v_weights[-1] / (1 - self.beta3**time)
            self.weights[-1] -= (self.alpha * m_w_hat ) / (np.sqrt(v_w_hat) + self.eta)

            # apply Adam updates to bias
            self.m_biases[-1] = self.beta2 * self.m_biases[-1] + (1 - self.beta2) * db
            self.v_biases[-1] = self.beta3 * self.v_biases[-1] + (1 - self.beta3) * (db ** 2)
            m_b_hat = self.m_biases[-1] / (1 - self.beta2**time)
            v_b_hat = self.v_biases[-1] / (1 - self.beta3**time)
            self.biases[-1] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

            # Propagate gradients backward through hidden layers
            for l in range(len(self.weights) - 2, -1, -1):
                delta = self._get_delta_update(delta, self.weights[l+1], zs[l]) #get delta activation for current layer
                deltas.append(delta)
                self.delta_history.append(deltas)
                dW_hidden = np.dot(activations[l].T, delta) / X.shape[0]
                db_hidden = np.sum(delta, axis=0) / X.shape[0]

                # apply Adam updates to weights
                self.m_weights[l] = self.beta2 * self.m_weights[l] + (1 - self.beta2) * dW_hidden
                self.v_weights[l] = self.beta3 * self.v_weights[l] + (1 - self.beta3) * (dW_hidden ** 2)
                m_w_hat = self.m_weights[l] / (1 - self.beta2**time)
                v_w_hat = self.v_weights[l] / (1 - self.beta3**time)
                self.weights[l] -= (self.alpha * m_w_hat) / (np.sqrt(v_w_hat) + self.eta)

                # apply Adam updates to bias
                self.m_biases[l] = self.beta2 * self.m_biases[l] + (1 - self.beta2) * db_hidden
                self.v_biases[l] = self.beta3 * self.v_biases[l] + (1 - self.beta3) * (db_hidden ** 2)
                m_b_hat = self.m_biases[l] / (1 - self.beta2**time)
                v_b_hat = self.v_biases[l] / (1 - self.beta3**time)
                self.biases[l] -= (self.alpha * m_b_hat) / (np.sqrt(v_b_hat) + self.eta)

            self.weights_history.append([w.copy() for w in self.weights])
            self.biases_history.append([b.copy() for b in self.biases])
            
        return self
        
            
    def _get_shuffled(self, n_samples, X, y):
        permutation = np.random.permutation(n_samples)
        X_shuffled = X[permutation]
        y_shuffled = y[permutation]
        return X_shuffled, y_shuffled
    
    def _set_epoch_loss(self, y, activation):
        epoch_loss = self._compute_loss(y, activation)
        self.loss_history.append(epoch_loss)
        return self
    
    def _store_preds(self, X, y, activations, t, epoch):
        preds = {}
        activations, _ = self._forward_pass(X)
        self._set_epoch_loss(y, activations[-1])
        preds[f'activation {t}, {epoch}'] = activations[-1].flatten()
        self.estimator_preds.append(preds) # store the preds
        return self
    
    def _fit_MLP(self, t, X, residuals): #activations are multipled by learning rate and alpha
        X = np.asarray(X)
        y = np.asarray(residuals).reshape(-1, 1) #changing to y for easy coding
        n_samples, n_features = X.shape
        if self.batch_size is None: self.batch_size = n_samples #if no batch, set to number of samples
        
        # Initialize parameters
        self._initialize_parameters(n_features)
        
        self.weights_history.append([w.copy() for w in self.weights])
        self.biases_history.append([b.copy() for b in self.biases])
        activations, _ = self._forward_pass(X)
        initial_loss = self._compute_loss(y, activations[-1])
        self.loss_history.append(initial_loss)
        
        # global time step for Adam bias correction
        time = 0
        
        # Training loop
        if self.optimiser is None:
            for epoch in range(self.n_epochs):
            # Shuffle data: -------------------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)
                # Mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i:i + self.batch_size]
                    y_batch = y_shuffled[i:i + self.batch_size]
                    # Forward pass
                    activations, zs = self._forward_pass(X_batch) #output for model prediction
                    # Backward pass
                    self._backward_pass(X_batch, y_batch, activations, zs)
                    self._store_preds(X_batch, y_batch, activations, t, epoch)
                #end loop---------------------------------------------------
        
        if self.optimiser is 'SGD':
            for epoch in range(self.n_epochs):
            # shuffle datasets
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)

                # mini-batch loop
                for i in range(0, n_samples, self.batch_size):
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]
                    #forward prop
                    activations, zs = self._forward_pass(X_batch)
                    #back prop
                    self._backward_pass(X_batch, y_batch, activations, zs)

                    #store results
                    self._store_preds(X_batch, y_batch, activations, t, epoch)

        if self.optimiser is 'adam':
             for epoch in range(self.n_epochs):
                # Shuffle data:  Gradient descent-----------------------------------
                X_shuffled, y_shuffled = self._get_shuffled(n_samples, X, y)
                
                #mini batch loop
                for i in range(0, n_samples, self.batch_size):
                    time += 1
                    X_batch = X_shuffled[i : i + self.batch_size]
                    y_batch = y_shuffled[i : i + self.batch_size]

                    # 1. forward prop
                    activations, zs = self._forward_pass(X_batch)
                    # Backward prop
                    self._backward_pass(X_batch, y_batch, activations, zs, time)
                    #store results
                    self._store_preds(X_batch, y_batch, activations, t, epoch)

        return self
    #=======================================Fit FUNCTIONS=======================================
    def _early_stopping(self, X, y, t):
        val_score = self.score(X, y)
        self.validation_scores.append(val_score)
        print('Current score: ', val_score, '\nBest score: ', self.best_val_score, '\nTolerance: ', self.best_val_score - self.tolerance)
        if val_score < self.best_val_score - self.tolerance:
            print('continue')
            self.best_val_score = val_score
            self.validation_scores.append(val_score)
            self.best_iteration = t + 1 #iterations are 0 inclusive, so need an extra 1
            self.no_improvement_count = 0 #reset the clock
        else: self.no_improvement_count += 1 #if going badly, lose patience
        
        self.n_estimators = self.best_iteration #update number of estimators so the prediction doesn't go awry

        if self.no_improvement_count >= self.patience:
            print(f'Early stopping at iteration {t+1}. Best validation {self.measure}: {self.best_val_score}')
            self.n_estimators = self.best_iteration #update number of estimators so the prediction doesn't go awry
            return True
            
        return False

    def _update_predictions(self, y_pred, feature_index, threshold, left_value, right_value, X):
        return np.array([y_pred[i] + (self.learning_rate * left_value) if X[:,feature_index][i] <= threshold else y_pred[i] + (self.learning_rate * right_value) for i in range(len(X[:,feature_index]))])  
    
    def fit(self, X, y):
        if self.validation_fraction is not None: #start early stopping
            X, X_val, y, y_val = modsel.train_test_split(X, y, test_size = self.validation_fraction, random_state = self.seed)
        
        self.initial_pred = self._initialize_predictions(y)
        self.current_pred = np.full(len(y), self.initial_pred)
                
        for t in range(self.n_estimators): #growing t trees... ba-dump, tiss
            print(f'fitting round {t}')
            print('fit self X and y len: ', len(X), len(y))
            print('fit self.current_pred len: ', len(self.current_pred)) 
            #========================  
            residuals = self._compute_residuals(y, self.current_pred) # get residuals from the current predictions
            #========================
            if self.classifier is 'stump': self._fit_stump(t, X, residuals)
            elif self.classifier is 'tree': self._fit_tree(t, X, residuals)
            elif self.classifier is 'MLP': self._fit_MLP(t, np.asarray(StandardScaler().fit_transform(X)), residuals)
            
#             print('PRINTING TREES: \n',self.trees)
            
            #check early stopping criteria
            if self.validation_fraction is not None: 
                print(f'checking eraly stopping at round {t}')
                stop = self._early_stopping(X_val, y_val, t)
                if stop is True: break
                
        return self
    
    def _predict_sample(self, x, node): #goes to bottom of tree to get prediction, will need to do update_predictions
        if node['is_leaf']: return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._predict_sample(x, node['left'])
        else: return self._predict_sample(x, node['right'])
    
    def predict(self, X):
        y_pred = np.full(len(X), self.initial_pred)
        
        if self.classifier is 'stump':
            for stump in self.stumps:
                y_pred = self._update_predictions(y_pred, stump[0], stump[1], stump[2], stump[3], X) #i managed to do this in one line because of list comprehension!
            return y_pred
          
        #iterate through each tree, get predictsion, weight them by learning rate and add the tree predictions to the total prediction (weight by gamme later)
        elif self.classifier is 'tree': 
            for t in range(len(self.trees)):
                print(f'at loop {t}, n_estimators = {self.n_estimators}')
                predictions = [] #np.zeros(len(X))
                print(f'printing tree {t}:', self.trees[t][f'tree {t}'])
                print(f'len trees {len(self.trees)}')
                current_tree = self.trees[t][f'tree {t}']
                
                for x in X:
                    predictions.append(self._predict_sample(x, current_tree))
                predictions = np.array(predictions) * self.learning_rate
                y_pred += predictions
        
        elif self.classifier is 'MLP':
            X = StandardScaler().fit_transform(X)
            for t in range(self.n_estimators):
                X = np.asarray(X)
                activations, _ = self._forward_pass(X)
                predictions = activations[-1].flatten()
                predictions *= self.learning_rate #updated during forward and back prop
                y_pred += predictions #add to y_pred
          
        return y_pred   
        
    def score(self, X, y): #for grid search and CV etc
        y_pred = self.predict(X)
        if self.measure is 'MSE': return np.mean((y - y_pred)**2)
        elif self.measure is 'RMSE': return np.sqrt(np.mean((y - y_pred) ** 2))
        elif self.measure is 'MAE': return np.mean(np.abs(y - y_pred))
        elif self.measure is 'MAPE': return np.mean(np.abs((y - y_pred) / y))
        elif self.measure is 'R2': return 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - np.mean(y)) ** 2))
        return 